In [ ]:
#| default_exp edit_interactive

## Edit-interactive plan execution

Run a tiny Lisette agent loop against one notebook at a time. The inner agent receives the user plan plus a single compact notebook view, then can mutate only that notebook through scoped tools.

Sometimes a user wants an agent to carry out a bounded notebook edit rather than manually choose each `write_nb` or `update_cell` call. This notebook builds that inner edit loop: one notebook, one plan, a small set of notebook-aware tools, and a final diff.

The edit loop is for bounded delegation, not open-ended repository work. Its job is to give an inner agent a stable notebook view, a small tool belt, and revision-aware feedback so a single notebook can be edited and reviewed without exposing raw notebook JSON.

```python
execute_plan("nbs/02_write.ipynb", "Add an example after the write_nb docs", max_steps=4)
```

## Foundation and contracts

The implementation starts with a narrow contract: one bounded notebook, explicit tool capabilities, and revision-aware feedback.

#### Production contract

The edit-interactive loop is experimental. It stays out of the production core unless it has focused contract tests for bounded scope, stable notebook views, deterministic tool results, final diffs, and clear failure behavior when an inner edit cannot be completed.

In [ ]:
#| hide
from contextlib import redirect_stdout
from io import StringIO
import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_tmp_nb, write_nb as _write_tmp_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
#| export
import ast
import inspect
import json
import os
import re
import signal
import traceback
import uuid
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path
from threading import Lock, current_thread, main_thread
from fastcore.nbio import read_nb
from lisette import StopResponse
from nbskill.edit import NotebookEditor
from nbskill.foundation import (
    cap_text, cell_source, chapter_spans, commit_notebook, parse_one_cell,
    source_hash,
)
from nbskill.graph import symbol_usage_summary
from nbskill.knowledge import problem_statement_add, problem_statement_query, reference_query
from nbskill.parallel import notebook_locks
from nbskill.review import diff_nb

In [ ]:
#| export
_AGENT_MAX_STEPS = 20
_AGENT_MAX_CONTEXT_STEPS = 5
_AGENT_MAX_CONTEXT_COMMANDS = 20
_AGENT_MAX_FEEDBACK_ROUNDS = 8
_AGENT_MAX_TIMEOUT_SECONDS = 120
_AGENT_MAX_MESSAGE_CHARS = 12000
_AGENT_MAX_PROJECT_CONTEXT_CHARS = 6000
_AGENT_MAX_NOTEBOOK_VIEW_CHARS = 80000
_AGENT_MAX_CELL_SOURCE_CHARS = 8000
_AGENT_MAX_LOG_ITEMS = 200
_AGENT_MAX_DIFF_CHARS = 50000
_AGENT_SESSIONS = {}

In [ ]:
#| export
def _bounded_int(value, default, minimum, maximum, name):
    "Coerce and clamp an integer option."
    if value is None: value = default
    try:
        value = int(value)
    except (TypeError, ValueError):
        raise ValueError(f"{name} must be an integer") from None
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}")
    return min(value, maximum)

In [ ]:
#| export
def _bounded_text(text, limit=_AGENT_MAX_MESSAGE_CHARS):
    "Coerce text and cap it to a safe message length."
    return cap_text(str(text or ""), limit)

In [ ]:
#| export
def _bounded_items(items, limit=_AGENT_MAX_LOG_ITEMS):
    "Return only the most recent bounded log items."
    return list(items)[-limit:]

In [ ]:
#| export
def _save_notebook(nb, path):
    "Commit a notebook after comparing it with the current file."
    return commit_notebook(path, nb, before=read_nb(path))

#### The inner-agent contract

The system prompt is intentionally narrow. The inner agent edits exactly one notebook, uses only the provided tools, prefers stable cell ids, and stops with a summary when the plan is done.

In [ ]:
#| export
EDIT_INTERACTIVE_SYSTEM = """You are an nbskill notebook-editing subagent.
You may edit one or more target notebooks in one repository. You receive the
project description, a focused knowledge summary, optional caller/callee impact
for the symbols in scope, and a managed notebook context prepared by a separate
context-management step.

Use only the provided tools: str_replace, edit_cell, add_cell, delete_cell,
run_code, inspect_state, execute_cell, query_knowledge, query_problem_memory,
record_problem_solution, inspect_project, inspect_library, and the context tools open_context, fold_context,
delete_context, edit_context, context_view, and manage_context. Prefer
manage_context to batch multiple context operations in one tool call. When more
than one notebook is in scope, pass the notebook path/name to edit and execution
tools so the target is explicit. Keep the work scoped to one small specific
task.

Problem-solving mindset: before choosing an approach, state the concrete
problem you are solving, call query_problem_memory for similar old problems,
compare the old solution with your current idea, and look for a simpler or
better way to solve this task. Be willing to dislike the existing code: if a
cell, helper, or approach is confusing, brittle, duplicated, or fighting the
problem, prefer deleting it and rewriting the small scoped behavior clearly over
preserving bad structure. A rewrite still needs a narrow target, a short
rationale, and focused verification. After the task, call record_problem_solution
for each reusable problem-solution pair you learned, with short evidence and at least four useful tags such as topic, sub-topic, library, problem-category, and solution-category.

For new behavior, use this loop:
1. Scratch: run the smallest experiment with run_code in the live notebook
   scope. Use budget_secs for code that might run long. Effectful scratch requests pause for user approval; preserve the session id and continue after the decision.
2. Inspect: use inspect_state to look at variables and functions before treating
   an experiment as correct.
3. Function: turn the working scratch into a focused function. If the current
   code is worse than the replacement, delete or rewrite the scoped cell instead
   of layering patches on top. Add a Markdown cell directly after exported code
   explaining why the function is needed.
4. Example: add an example cell directly below the function showing how it
   works. If the example is slow or produces artifacts, add `#| eval: false`.
5. Test: add a focused test cell directly below the example so future changes
   keep the same output.
6. Execute: use execute_cell to replay through the example or test. Pass
   rerun_all=True when earlier cells changed and the whole notebook state must
   be rebuilt.
7. Clean context: after scratch is stabilized into notebook cells, fold or
   delete no-longer-needed working context.

After a notebook or context mutation, the tool loop may stop and return a short
feedback note. Read that note, decide whether the change looks correct, then
continue, fix, test, or fold context. Exceptions are returned as tool output;
inspect them, edit, and execute again. Stop as soon as the requested notebook
change is complete. Your final message must summarize what changed, what was
executed, and what could not be completed.
"""

CONTEXT_SESSION_SYSTEM = """You manage the context for an nbskill editing run.
You do not edit notebooks. Use the context tools to open, fold, delete, or edit
managed context messages so the next editing step sees the smallest useful
context. Prefer manage_context to batch multiple operations in one tool call,
e.g. [{"action":"open","tag":"ch3"},{"action":"fold","tag":"ch1","content":"summary"}].
Open the few chapters directly relevant to the plan, fold full chapters to
concise summaries after they are no longer useful, delete irrelevant material
from the active context while preserving the audit log, and leave the index
visible. Stop once the context is prepared or cleaned.
"""

In [ ]:
#| exporti
_CAPTURE_LOCK = Lock()

In [ ]:
#| export
def capture_call_text(func, **kwargs):
    "Run `func` and return captured stdout/stderr, or the return value."
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(str(result))
    return cap_text("\n".join(chunk for chunk in chunks if chunk), _AGENT_MAX_MESSAGE_CHARS)

The public `capture_call_text` helper turns a function call into compact text, which keeps tool feedback readable.

In [ ]:
print(capture_call_text(lambda: "captured"))

In [ ]:
#| hide
assert capture_call_text(lambda: "captured") == "captured"

## Notebook views and managed context

Stable views and tagged context let an agent inspect the right notebook without losing cell identity.

#### A stable notebook view

The edit loop needs a text representation that is compact enough for a model but precise enough for safe edits. `notebook_view` includes ids, cell types, and full cell source.

In [ ]:
#| export
def notebook_view(
    path,  # Notebook path to render
    revision=0,  # Revision number to include in the rendered view
):
    "Render one notebook as a compact, stable text view."
    path = Path(path)
    with notebook_locks(path):
        nb = read_nb(path)
        lines = [f"Notebook: {path}", f"Revision: {revision}", ""]
        for idx, cell in enumerate(nb.cells):
            lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
            lines.append("<<<SOURCE")
            lines.append(cap_text(cell_source(cell).rstrip(), _AGENT_MAX_CELL_SOURCE_CHARS))
            lines.append("SOURCE")
            lines.append("")
        return cap_text("\n".join(lines).rstrip() + "\n", _AGENT_MAX_NOTEBOOK_VIEW_CHARS)

A compact view exposes the notebook path and stable cell ids before an agent edits anything.

In [ ]:
with write_demo_notebook("08_view_example.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("## Demo\nA small chapter.")]), path)
    print(notebook_view(path).splitlines()[0])

In [ ]:
#| hide
with write_demo_notebook("08_view_test.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("## Demo")]), path)
    view = notebook_view(path)
    assert "Notebook:" in view
    assert "CELL 0" in view

In [ ]:
#| export
def _split_notebooks(notebooks):
    "Normalize notebook selectors into non-empty path strings."
    if notebooks is None: return []
    if isinstance(notebooks, (str, Path)): return [item.strip() for item in str(notebooks).split(",") if item.strip()]
    return [str(item) for item in notebooks if str(item).strip()]

In [ ]:
#| export
def _target_paths(notebooks):
    "Resolve notebook selectors into unique target paths."
    paths = [Path(item) for item in _split_notebooks(notebooks)]
    if not paths: raise ValueError("At least one target notebook is required.")
    seen, duplicates = set(), []
    for path in paths:
        key = path.as_posix()
        if key in seen: duplicates.append(key)
        seen.add(key)
    if duplicates: raise ValueError(f"Duplicate notebook target(s): {', '.join(sorted(duplicates))}")
    return paths

In [ ]:
#| export
def _target_slug(path):
    "Return a stable slug for a target notebook path."
    rel = Path(path).with_suffix("").as_posix()
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", rel).strip("-.") or "notebook"

In [ ]:
#| export
def _target_label(paths):
    "Return a readable label for one or more target paths."
    paths = [Path(path) for path in paths]
    if len(paths) == 1: return str(paths[0])
    return ", ".join(str(path) for path in paths)

In [ ]:
#| export
def _resolve_target_path(paths, notebook=None):
    "Resolve a user notebook selector against session targets."
    paths = [Path(path) for path in paths]
    name = _none_if_blank(notebook)
    if name is None:
        if len(paths) == 1: return paths[0]
        raise ValueError("notebook is required when an edit session has multiple target notebooks")
    matches = []
    for path in paths:
        choices = {str(path), path.as_posix(), path.name, path.stem, _target_slug(path)}
        if str(name) in choices: matches.append(path)
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"Unknown target notebook {name!r}; choose one of: {_target_label(paths)}")
    raise ValueError(f"Ambiguous target notebook {name!r}")

In [ ]:
#| export
def notebooks_view(
    paths,  # Notebook paths to render
    revision=0,  # Revision number to include in each rendered view
):
    "Render one or more notebooks as compact stable text."
    return "\n".join(notebook_view(path, revision=revision).rstrip() for path in _target_paths(paths)) + "\n"

## Editing and execution

The editing tools and runtime keep notebook mutations observable, scoped, and replayable.

#### Self-managed notebook context

A managed context keeps the same stable notebook addressing as `notebook_view`, but starts with only a chapter index. The inner agent can open, hide, remove, or edit tagged context messages by emitting compact text commands.

In [ ]:
#| export
class ManagedContextMessage:
    "One mutable context message controlled by a stable tag."
    def __init__(self, tag, purpose, content, visible=True, removed=False, summary=""):
        "Store the full and rendered state for one context message."
        self.tag = str(tag)
        self.purpose = str(purpose)
        self.content = str(content)
        self.visible = bool(visible)
        self.removed = bool(removed)
        self.summary = str(summary or "")

In [ ]:
#| export
def _context_msg_get(msg, key):
    "Read one field from a dict or object context message."
    return msg.get(key) if isinstance(msg, dict) else getattr(msg, key)

In [ ]:
#| export
def _context_msg_set(msg, key, value):
    "Write one field on a dict or object context message."
    if isinstance(msg, dict): msg[key] = value
    else: setattr(msg, key, value)

In [ ]:
#| export
def _chapter_tag(cell, path=None):
    "Return the stable managed-context tag for a chapter cell."
    base = f"chapter:{getattr(cell, 'id', '')}"
    return base if path is None else f"notebook:{_target_slug(path)}:{base}"

In [ ]:
#| export
def _public_symbols_in_source(source):
    "Return public definitions found in a source string."
    try: tree = ast.parse(source)
    except SyntaxError: return []
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and not node.name.startswith("_"):
            symbols.append(node.name)
    return symbols

In [ ]:
#| export
def _chapter_public_symbols(cells):
    "Return public definitions found across chapter cells."
    symbols = []
    for cell in cells:
        if getattr(cell, "cell_type", None) == "code": symbols.extend(_public_symbols_in_source(cell_source(cell)))
    return symbols

In [ ]:
#| export
def _first_markdown_paragraph(cells):
    "Return the first prose paragraph from markdown cells."
    for cell in cells:
        if getattr(cell, "cell_type", None) != "markdown": continue
        chunks, current = [], []
        for line in cell_source(cell).splitlines():
            stripped = line.strip()
            if stripped.startswith("#") or stripped.startswith("#|"): continue
            if stripped: current.append(stripped)
            elif current:
                chunks.append(" ".join(current))
                current = []
        if current: chunks.append(" ".join(current))
        if chunks: return cap_text(chunks[0], 360)
    return "No prose summary found."

In [ ]:
#| export
def _chapter_summary(title, cells):
    "Build a concise fold summary for one chapter."
    symbols = _chapter_public_symbols(cells)
    parts = [f"{title}: {_first_markdown_paragraph(cells)}", f"cells={len(cells)}"]
    if symbols: parts.append("symbols=" + ", ".join(symbols[:8]))
    return " | ".join(parts)

In [ ]:
#| export
def _chapter_source_view(path, span, cells):
    "Render the full source view for one chapter span."
    lines = [f"Full chapter context: {path}", f"Title: {span['title']}", f"Cells: {span['start']}:{span['end']}", ""]
    for idx in range(span["start"], span["end"]):
        cell = cells[idx]
        lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
        lines.append("<<<SOURCE")
        lines.append(cap_text(cell_source(cell).rstrip(), _AGENT_MAX_CELL_SOURCE_CHARS))
        lines.append("SOURCE")
        lines.append("")
    return cap_text("\n".join(lines).rstrip() + "\n", _AGENT_MAX_NOTEBOOK_VIEW_CHARS)

In [ ]:
#| export
def managed_notebook_context(
    path,  # Notebook path or paths to render into managed messages
    revision=0,  # Revision number included in the context index
    previous=None,  # Previous messages whose fold/delete state should be preserved
):
    "Build mutable summary-first context messages for one or more target notebooks."
    paths = _target_paths(path)
    multi = len(paths) > 1
    old = {msg.tag: msg for msg in (previous or [])}
    index_lines = [
        "Managed notebook context",
        "Notebooks: " + _target_label(paths),
        f"Revision: {revision}",
        "",
        "Open messages render full content; folded messages render summaries; deleted messages are omitted.",
        "",
        "Chapter index:",
    ]
    messages = []
    for path in paths:
        with notebook_locks(path):
            nb = read_nb(path)
            spans = chapter_spans(nb.cells, levels=range(1, 7), fallback="Notebook")
            if multi:
                index_lines.append("")
                index_lines.append(f"Notebook: {path}")
            for span in spans:
                cells = list(nb.cells[span["start"]:span["end"]])
                tag = _chapter_tag(nb.cells[span["start"]], path if multi else None)
                summary = _chapter_summary(span["title"], cells)
                index_lines.append(f"- {tag} title={span['title']!r} cells={span['start']}:{span['end']} summary={summary}")
                messages.append(ManagedContextMessage(
                    tag=tag,
                    purpose=f"full chapter: {path} :: {span['title']}",
                    content=_chapter_source_view(path, span, nb.cells),
                    visible=False,
                    summary=summary,
                ))
    index = ManagedContextMessage(
        "notebook:index", "chapter summary index", "\n".join(index_lines).rstrip() + "\n",
        summary="Index of available notebook context messages.",
    )
    rebuilt = [index, *messages]
    for msg in rebuilt:
        prior = old.get(msg.tag)
        if prior is None: continue
        prior_summary = getattr(prior, "summary", "")
        if msg.tag != "notebook:index":
            msg.visible, msg.removed = prior.visible, prior.removed
            if prior_summary: msg.summary = prior_summary
    return rebuilt

In [ ]:
#| export
def render_managed_context(
    messages,  # Managed context messages to project into prompt text
):
    "Render active managed context messages as a budgeted projection."
    visible = [msg for msg in messages if msg.visible and not msg.removed]
    folded = [msg for msg in messages if not msg.visible and not msg.removed]
    lines = ["Self-managed context messages", ""]
    for msg in visible:
        lines.append(f"<message tag={msg.tag!r} purpose={msg.purpose!r}>")
        lines.append(msg.content.rstrip())
        lines.append("</message>")
        lines.append("")
    if folded:
        lines.append("Folded messages:")
        for msg in folded:
            summary = (msg.summary or msg.purpose).rstrip()
            lines.append(f"<folded-message tag={msg.tag!r} purpose={msg.purpose!r}>")
            lines.append(summary)
            lines.append("</folded-message>")
            lines.append("")
    return "\n".join(lines).rstrip() + "\n"

Managed context keeps a chapter index visible and lets later commands open or fold individual chapters.

In [ ]:
with write_demo_notebook("08_context_example.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("## Demo\nManaged context.", cell_type="markdown")]), path)
    print(render_managed_context(managed_notebook_context(path)).splitlines()[0])

In [ ]:
#| hide
with write_demo_notebook("08_context_test.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("## Demo", cell_type="markdown")]), path)
    rendered = render_managed_context(managed_notebook_context(path))
    assert "notebook:index" in rendered

#### Session state

`EditSession` tracks the notebook path, revision, timeout, and operation logs. The revision count makes it clear which tool calls changed the notebook during a plan.

In [ ]:
#| export
@dataclass
class EditSession:
    "Mutable state for one edit-interactive notebook run."
    path: Path | str | list
    target_paths: list[Path] = field(default_factory=list)
    timeout: int = 30
    revision: int = 0
    session_id: str | None = None
    agent_id: str | None = None
    log_path: Path | None = None
    log: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)
    events: list[dict] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    managed_context: list[ManagedContextMessage] = field(default_factory=list)
    context_command_log: list[dict] = field(default_factory=list)
    feedback_notes: list[str] = field(default_factory=list)
    pending_feedback: str = ""
    live_ns: dict = field(default_factory=lambda: {"__name__": "__main__"})
    live_ns_by_path: dict = field(default_factory=dict)
    executed_until_idx: int = -1
    executed_until_by_path: dict = field(default_factory=dict)
    chat: object | None = None
    notebook_msg_idx: int | None = None
    context_msg_idx: int | None = None
    def __post_init__(self):
        "Normalize target paths and initialize session identifiers."
        paths = self.target_paths or _target_paths(self.path)
        self.target_paths = [Path(path) for path in paths]
        self.path = self.target_paths[0]
        if self.session_id is None:
            self.session_id = f"session-{uuid.uuid4().hex[:8]}"
        if self.agent_id is None:
            label = f"{_target_label(self.target_paths)}-{self.session_id}"
            self.agent_id = _agent_notebook_id(label)
        if self.log_path is None: self.log_path = Path("log") / f"agent-{self.agent_id}.log"
    def target_path(self, notebook=None):
        "Resolve a user-facing notebook selector against this session."
        return _resolve_target_path(self.target_paths, notebook)
    def _state_key(self, path):
        "Return the dictionary key used for live state by path."
        return Path(path).as_posix()
    def live_state(self, path):
        "Return the persistent Python namespace for one notebook."
        key = self._state_key(path)
        if key not in self.live_ns_by_path:
            self.live_ns_by_path[key] = {"__name__": "__main__"}
        return self.live_ns_by_path[key]
    def executed_until(self, path):
        "Return the last executed cell index for one notebook."
        return self.executed_until_by_path.get(self._state_key(path), -1)
    def set_executed_until(self, path, idx):
        "Record the last executed cell index for one notebook."
        self.executed_until_by_path[self._state_key(path)] = idx
    def reset_live_state(self, path=None):
        "Reset the live notebook execution namespace."
        if path is None:
            self.live_ns = {"__name__": "__main__"}
            self.live_ns_by_path = {}
            self.executed_until_idx = -1
            self.executed_until_by_path = {}
            return
        self.live_ns_by_path[self._state_key(path)] = {"__name__": "__main__"}
        self.executed_until_by_path[self._state_key(path)] = -1
    def update_context_view(self):
        "Render the current managed context into the active chat history."
        if self.chat is None or self.context_msg_idx is None: return
        if self.context_msg_idx >= len(self.chat.hist): return
        msg = self.chat.hist[self.context_msg_idx]
        _context_msg_set(msg, "content", render_managed_context(self.managed_context))
    def refresh_view(self):
        "Refresh notebook-derived context and replace the active chat view."
        if self.chat is None: return
        if self.managed_context and self.context_msg_idx is not None:
            self.managed_context = managed_notebook_context(
                self.target_paths, self.revision, previous=self.managed_context
            )
            self.update_context_view()
            return
        if self.notebook_msg_idx is None or self.notebook_msg_idx >= len(self.chat.hist): return
        msg = self.chat.hist[self.notebook_msg_idx]
        _context_msg_set(msg, "content", notebooks_view(self.target_paths, self.revision))
    def record_message(self, role, content):
        "Append one agent message to memory and the run log."
        item = {"revision": self.revision, "role": role, "content": _bounded_text(content)}
        self.messages.append(item)
        _append_agent_log(self, "message", item)
    def session_note(self, content):
        "Insert a feedback note into memory, active chat history, and the audit log."
        note = _bounded_text(content)
        self.pending_feedback = note
        self.feedback_notes.append(note)
        item = {"revision": self.revision, "role": "note", "content": note}
        self.messages.append(item)
        _append_agent_log(self, "feedback_note", item)
        if self.chat is not None:
            self.chat.hist.append({"role": "user", "content": note})
        return note
    def consume_feedback(self):
        "Return and clear the feedback note that should seed the next round."
        note = self.pending_feedback
        self.pending_feedback = ""
        return note
    def feedback_packet(self, result, message, path=None):
        "Build one compact notebook/context mutation feedback packet."
        result = result if isinstance(result, dict) else {}
        affected, diff_chunks = [], []
        for item in result.get("diffs", []):
            affected.extend(item.get("affected_cell_ids", []) or [])
            affected.extend(item.get("inserted_cell_ids", []) or [])
            if item.get("cell_id"): affected.append(item.get("cell_id"))
            if item.get("diff"): diff_chunks.append(str(item["diff"]))
        affected = sorted({str(item) for item in affected if item})
        detail = str(result.get("text") or "").strip()
        if detail and not diff_chunks: diff_chunks.append(detail)
        machine = {
            "status": "changed",
            "revision": self.revision,
            "notebook": str(path or self.path),
            "affected_cell_ids": affected,
        }
        chunks = [message, "tool_result=" + json.dumps(machine, sort_keys=True)]
        if affected: chunks.append("affected_cell_ids=" + ", ".join(affected))
        if diff_chunks:
            chunks.extend(["", "concise_diff:", cap_text("\n\n".join(diff_chunks), 3000)])
        chunks.extend([
            "",
            "Notebook/context updated. Continue with the next step if the change looks correct; otherwise fix it.",
        ])
        return _bounded_text("\n".join(chunks))
    def record(self, message):
        "Append an operation to the session log."
        self.log.append(f"r{self.revision}: {message}")
        _append_agent_log(self, "operation", {"revision": self.revision, "message": message})
    def record_tool(self, name, detail=""):
        "Append one tool use to the session history."
        detail = _bounded_text(detail, 1000) if detail else ""
        suffix = f"({detail})" if detail else "()"
        self.tool_log.append(f"r{self.revision}: {name}{suffix}")
        item = {"revision": self.revision, "tool": name}
        if detail: item["detail"] = detail
        self.history.append(item)
        _append_agent_log(self, "tool", item)
    def record_event(self, kind, **data):
        "Append one structured empirical event to the session audit log."
        item = {"revision": self.revision, "kind": kind}
        for key, value in data.items():
            if value is None: continue
            item[key] = _bounded_text(value, 1000) if isinstance(value, str) else value
        self.events.append(item)
        _append_agent_log(self, "event", item)
        return item
    def record_context_command(self, command, status="ok", detail=""):
        "Append one managed-context command to the session history."
        item = {"revision": self.revision, "status": status, **command}
        if detail: item["detail"] = _bounded_text(detail, 1000)
        self.context_command_log.append(item)
        _append_agent_log(self, "context_command", item)

In [ ]:
#| export
_CTX_SIMPLE_RE = re.compile(r"\[\[ctx:(open|delete)\s+([^\]]+)\]\]")
_CTX_BLOCK_RE = re.compile(r"\[\[ctx:(fold|edit)\s+([^\]]+)\]\](.*?)\[\[/ctx:\1\]\]", re.S)

In [ ]:
#| export
def find_context_command(
    text,  # Text stream to search for a managed-context command
):
    "Return the first complete managed-context command in `text`."
    simple = _CTX_SIMPLE_RE.search(text)
    block = _CTX_BLOCK_RE.search(text)
    matches = [match for match in [simple, block] if match is not None]
    if not matches: return None
    match = min(matches, key=lambda item: item.start())
    if match.re is _CTX_BLOCK_RE:
        action, tag, content = match.group(1), match.group(2).strip(), match.group(3)
    else:
        action, tag, content = match.group(1), match.group(2).strip(), None
    return {
        "action": action,
        "tag": tag,
        "content": content,
        "start": match.start(),
        "end": match.end(),
        "raw": match.group(0),
    }

In [ ]:
#| export
def _managed_message_by_tag(messages, tag):
    "Return the unique managed context message for a tag."
    matches = [msg for msg in messages if msg.tag == tag]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No managed context message has tag {tag!r}")
    raise ValueError(f"Multiple managed context messages have tag {tag!r}")

In [ ]:
#| export
def _context_feedback_note(action, tag):
    "Build the standard feedback note for a context command."
    return (
        f"context {action} applied to {tag}\n"
        "Notebook/context updated. Continue with the next step if the change "
        "looks correct; otherwise fix it."
    )

In [ ]:
#| export
def apply_context_command(
    session,  # Edit session whose managed context should change
    command,  # Parsed managed-context command to apply
):
    "Apply one managed-context command to an edit session."
    command = dict(command)
    tag, action = command.get("tag", ""), command.get("action", "")
    msg = _managed_message_by_tag(session.managed_context, tag)
    if action == "open":
        msg.visible, msg.removed = True, False
    elif action == "fold":
        summary = _bounded_text(command.get("content") or msg.summary or msg.purpose, _AGENT_MAX_MESSAGE_CHARS)
        msg.summary = summary
        msg.visible, msg.removed = False, False
    elif action == "delete":
        msg.removed, msg.visible = True, False
    elif action == "edit":
        content = _bounded_text(command.get("content") or "", _AGENT_MAX_NOTEBOOK_VIEW_CHARS)
        msg.content = content
        if not msg.purpose.startswith("edited "):
            msg.purpose = "edited " + msg.purpose
        if content.strip(): msg.summary = cap_text(" ".join(content.split()), 600)
        msg.visible, msg.removed = True, False
    else:
        raise ValueError(f"Unknown managed context action {action!r}")
    public = {key: command.get(key) for key in ("action", "tag")}
    session.record_context_command(public)
    session.update_context_view()
    note = _context_feedback_note(action, tag)
    session.session_note(note)
    return StopResponse(note)

In [ ]:
#| export
def _stream_chunk_text(chunk):
    "Extract text from a streamed chat chunk."
    if isinstance(chunk, str): return chunk
    if isinstance(chunk, bytes): return chunk.decode("utf-8", errors="replace")
    try:
        content = chunk.choices[0].delta.content
        return "" if content is None else str(content)
    except (AttributeError, IndexError, TypeError):
        pass
    try:
        content = chunk.choices[0].message.content
        return "" if content is None else str(content)
    except (AttributeError, IndexError, TypeError):
        return ""

In [ ]:
#| export
def _close_stream(stream):
    "Close a stream object when it supports close()."
    close = getattr(stream, "close", None)
    if callable(close): close()

In [ ]:
#| export
def _chat_response_text(response):
    "Return assistant text from a Lisette response-like object."
    if isinstance(response, str): return response
    text = _stream_chunk_text(response)
    if text: return text
    return str(response or "")

In [ ]:
#| export
def _chat_allows_nonstream(chat):
    "Return whether this model can recover with a non-streaming call."
    model = str(getattr(chat, "model", ""))
    return not model.startswith("chatgpt/")

In [ ]:
#| export
def _chat_nonstream(session, prompt, max_steps):
    "Run one non-streaming continuation when provider streaming fails."
    if not _chat_allows_nonstream(session.chat):
        raise ValueError(f"{session.chat.model} requires stream=True; cannot use non-stream fallback")
    response = session.chat(prompt, max_steps=max_steps, stream=False)
    return _chat_response_text(response)

In [ ]:
#| export
def run_chat_with_context_commands(
    session,  # Edit session with an attached chat and managed context
    prompt,  # Prompt to send to the chat model
    max_steps=8,  # Maximum model tool steps for each chat call
    max_context_commands=8,  # Maximum context commands to intercept
):
    "Run a streaming chat, intercepting managed-context text commands."
    if session.chat is None: raise ValueError("session.chat is required")
    max_steps = _bounded_int(max_steps, 8, 1, _AGENT_MAX_STEPS, "max_steps")
    max_context_commands = _bounded_int(max_context_commands, 8, 0, _AGENT_MAX_CONTEXT_COMMANDS, "max_context_commands")
    output, next_prompt = [], prompt
    for _ in range(max_context_commands + 1):
        stream = None
        try:
            stream = session.chat(next_prompt, max_steps=max_steps, stream=True)
            buffer, command = "", None
            for chunk in stream:
                piece = _stream_chunk_text(chunk)
                if not piece: continue
                buffer = cap_text(buffer + piece, _AGENT_MAX_MESSAGE_CHARS)
                command = find_context_command(buffer)
                if command:
                    _close_stream(stream)
                    break
        except BaseException as exc:
            _close_stream(stream)
            if not session.context_command_log: raise
            detail = f"{type(exc).__name__}: {exc}"
            if not _chat_allows_nonstream(session.chat): raise
            session.record_context_command(
                {"action": "fallback", "tag": "managed-context"},
                status="ok",
                detail=detail,
            )
            output.append(_chat_nonstream(session, next_prompt, max_steps=max_steps))
            return "".join(output).strip()
        if command is None:
            output.append(buffer)
            return "".join(output).strip()
        output.append(buffer[:command["start"]])
        note = str(apply_context_command(session, command))
        next_prompt = (
            note + "\nContinue the same task from just before the command. "
            "Do not repeat the command."
        )
    session.record_context_command({"action": "limit", "tag": "managed-context"}, status="error", detail="max_context_commands exceeded")
    return "".join(output).strip() or "managed context command limit reached"

In [ ]:
#| export
def _none_if_blank(value):
    "Normalize blank strings to None."
    if value is None: return None
    value = str(value)
    return None if value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _edit_find_cell_by_id(cells, cell_id):
    "Return the index and cell for a notebook cell id."
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == str(cell_id)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No cell has id {cell_id!r}")
    raise ValueError(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def _result_cell_ids(result):
    "Return affected or inserted cell ids from a notebook edit result."
    result = result if isinstance(result, dict) else {}
    affected = []
    for item in result.get("diffs", []):
        affected.extend(item.get("affected_cell_ids", []) or [])
        affected.extend(item.get("inserted_cell_ids", []) or [])
        if item.get("cell_id"): affected.append(item.get("cell_id"))
    return sorted({str(item) for item in affected if item})

In [ ]:
#| export
def _finish_write(session, path, message, result, tool_name=None):
    "Finalize a notebook mutation and return feedback to the agent."
    session.revision += 1
    session.reset_live_state(path)
    session.record(message)
    status = "changed" if not isinstance(result, dict) or result.get("changed", True) else "no_change"
    session.record_event(
        "write", tool=tool_name, notebook=str(path), status=status,
        affected_cell_ids=_result_cell_ids(result),
    )
    session.refresh_view()
    note = session.feedback_packet(result, message, path=path)
    if status == "changed":
        note = _bounded_text(
            note
            + "\n\nEmpirical next step: inspect or run the changed behavior, then execute the focused example/test."
        )
    session.session_note(note)
    return StopResponse(note)

In [ ]:
#| export
def _agent_notebook_id(path):
    "Build a filesystem-safe id for an agent notebook run."
    rel = Path(path).with_suffix("").as_posix()
    rel = re.sub(r"[^A-Za-z0-9_.-]+", "-", rel).strip("-.")
    return rel or "notebook"

In [ ]:
#| export
def _append_agent_log(session, kind, payload):
    "Append one JSONL audit event for an edit session."
    path = Path(session.log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "kind": kind,
        "agent_id": session.agent_id,
        "notebook": str(session.path),
        "target_notebooks": [str(path) for path in session.target_paths],
        "payload": payload,
    }
    path.open("a", encoding="utf-8").write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
#| export
def _project_description(path):
    "Return a compact repository description for prompts."
    start = Path(path).resolve().parent
    for root in [start, *start.parents]:
        readme = root / "README.md"
        if readme.exists(): return cap_text(readme.read_text(encoding="utf-8"), 2400)
    return "No README.md project description was found."

In [ ]:
#| export
def _knowledge_context(query, top_k=3):
    "Return a compact reference-knowledge summary for a query."
    try: result = reference_query(query, top_k=top_k, current_repo=".")
    except BaseException as exc: return f"Knowledge query unavailable: {type(exc).__name__}: {exc}"
    hits = []
    for hit in result.get("hits", []):
        label = ".".join(item for item in [hit.get("module"), hit.get("symbol")] if item)
        source = cap_text(hit.get("source") or hit.get("docstring") or "", 700)
        hits.append(f"- {label or hit.get('path')}: {source}")
    return "\n".join(hits) if hits else "No relevant knowledge hits."

In [ ]:
#| export
def _problem_memory_context(query, top_k=5):
    "Return compact reusable problem-solution memories for a query."
    try: result = problem_statement_query(query, top_k=top_k)
    except BaseException as exc: return f"Problem memory unavailable: {type(exc).__name__}: {exc}"
    return result.get("context") or "No similar problem memories found."

In [ ]:
#| export
def _symbol_impact_context(symbols):
    "Return a compact symbol impact summary for scoped symbols."
    names = _split_notebooks(symbols)
    if not names: return "No symbols requested for caller/callee impact."
    try: return symbol_usage_summary(".", names)
    except BaseException as exc: return f"Symbol impact unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def _subagent_context(path, plan, symbols=None):
    "Build the project context injected into the editing agent."
    return "\n\n".join([
        "Project description:\n" + _project_description(path),
        "Knowledge summary:\n" + _knowledge_context(plan),
        "Problem memory:\n" + _problem_memory_context(plan),
        "Caller/callee impact:\n" + _symbol_impact_context(symbols),
    ])

#### The editing tool belt

The inner loop only gets four notebook operations: add, edit, run through a cell, and remove. Each operation validates inputs, refreshes the notebook view, and records a diff-like message for the final report.

## Runtime approvals

Scratch code can keep state, while effectful capabilities pause for an explicit approval.

In [ ]:
#| export
def _inserted_cell_ids(result):
    "Return inserted cell ids from edit_notebook diffs."
    inserted = []
    for diff in result.get("diffs", []):
        inserted.extend(diff.get("inserted_cell_ids", []))
    return inserted

In [ ]:
#| export
def _similar_source_lines(source, needle, limit=6):
    "Return source lines that partially match a stale edit string."
    words = {word for word in re.findall(r"\w+", str(needle or "")) if len(word) > 2}
    scored = []
    for number, line in enumerate(str(source or "").splitlines(), 1):
        score = sum(1 for word in words if word in line)
        if score:
            scored.append((score, number, line.strip()))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [f"L{number}: {cap_text(line, 240)}" for _, number, line in scored[:limit]]

In [ ]:
#| export
def _edit_mismatch_feedback(path, id, before, old_str, matches):
    "Return a no-change report for an edit_cell match failure."
    machine = {
        "status": "no_change",
        "reason": "old_str_match_count",
        "matches": matches,
        "notebook": str(path),
        "cell_id": id,
        "source_hash": source_hash(before),
    }
    chunks = [
        "edit_cell made no change because old_str did not match exactly once.",
        "tool_result=" + json.dumps(machine, sort_keys=True),
    ]
    similar = _similar_source_lines(before, old_str)
    if similar:
        chunks.extend(["", "similar_lines:", "\n".join(similar)])
    chunks.extend(["", "current_source:", cap_text(before, _AGENT_MAX_CELL_SOURCE_CHARS)])
    return "\n".join(chunks)

In [ ]:
#| export
def make_edit_tools(
    session,  # Edit session the returned tools should mutate or inspect
):
    "Create notebook-scoped editing tools for one edit-interactive session."

    def str_replace(old_str: str, new_str: str, notebook: str | None = None) -> str:
        "Replace one exact string occurrence anywhere in a target notebook."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            matches = []
            for idx, cell in enumerate(nb.cells):
                source = cell_source(cell)
                if old_str in source:
                    matches.append((idx, cell, source))
        session.record_tool("str_replace", f"notebook={path}, matches={len(matches)}")
        if len(matches) != 1:
            raise ValueError(
                f"old_str matched {len(matches)} cells in {path}; expected exactly 1"
            )
        idx, cell, before = matches[0]
        after = before.replace(old_str, new_str, 1)
        result = NotebookEditor(path, auto_feedback=False).replace_cell(
            cell.id, after, cell_type=cell.cell_type
        )
        msg = f"Replaced text in {path} id={cell.id}"
        return _finish_write(session, path, msg, result, tool_name="str_replace")

    def edit_cell(
        id: str, old_str: str, new_str: str, notebook: str | None = None
    ) -> str:
        "Replace one exact string occurrence inside one cell."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            before = cell_source(cell)
            count = before.count(old_str)
        session.record_tool("edit_cell", f"notebook={path}, id={id!r}, matches={count}")
        if count != 1:
            return _edit_mismatch_feedback(path, id, before, old_str, count)
        after = before.replace(old_str, new_str, 1)
        result = NotebookEditor(path, auto_feedback=False).replace_cell(
            id, after, cell_type=cell.cell_type
        )
        msg = f"Edited text in {path} id={id}"
        return _finish_write(session, path, msg, result, tool_name="edit_cell")

    def add_cell(
        after_id: str | None = None,
        content: str = "",
        notebook: str | None = None,
    ) -> str:
        "Add one cell to a target notebook."
        path = session.target_path(notebook)
        anchor = _none_if_blank(after_id)
        new_cell = parse_one_cell(content, "code")
        if anchor is not None:
            with notebook_locks(path):
                _edit_find_cell_by_id(read_nb(path).cells, anchor)
        session.record_tool("add_cell", f"notebook={path}, after_id={anchor!r}")
        result = NotebookEditor(path, auto_feedback=False).insert(
            anchor, cell_source(new_cell), cell_type=new_cell.cell_type
        )
        inserted = _inserted_cell_ids(result)
        new_id = inserted[0] if inserted else ""
        where = f"after id={anchor}" if anchor is not None else "at end"
        msg = f"Added cell id={new_id} to {path} {where}"
        return _finish_write(session, path, msg, result, tool_name="add_cell")

    def delete_cell(id: str, notebook: str | None = None) -> str:
        "Delete one cell from a target notebook."
        path = session.target_path(notebook)
        with notebook_locks(path):
            _edit_find_cell_by_id(read_nb(path).cells, id)
        session.record_tool("delete_cell", f"notebook={path}, id={id!r}")
        result = NotebookEditor(path, auto_feedback=False).delete(id)
        msg = f"Deleted cell id={id} from {path}"
        return _finish_write(session, path, msg, result, tool_name="delete_cell")

    def _exec_python(source, filename, ns):
        "Execute Python source and evaluate a final expression when present."
        tree = ast.parse(source, filename=filename, mode="exec")
        if not tree.body:
            return None
        if not isinstance(tree.body[-1], ast.Expr):
            exec(compile(tree, filename, "exec"), ns)
            return None
        prefix = ast.Module(body=tree.body[:-1], type_ignores=[])
        expr = ast.Expression(tree.body[-1].value)
        ast.fix_missing_locations(prefix)
        ast.fix_missing_locations(expr)
        if prefix.body:
            exec(compile(prefix, filename, "exec"), ns)
        return eval(compile(expr, filename, "eval"), ns)

    def _eval_false(source):
        "Return whether a cell source disables evaluation."
        return any(
            re.match(r"#\|\s*eval:\s*false\b", line.strip(), re.I)
            for line in source.splitlines()[:5]
        )

    def _time_limit_supported():
        "Return whether signal-based time limits can run here."
        return (
            hasattr(signal, "setitimer")
            and hasattr(signal, "SIGALRM")
            and current_thread() is main_thread()
        )

    @contextmanager
    def _time_limit(seconds):
        "Apply a signal timer for a bounded code block."
        def _raise_timeout(signum, frame):
            "Raise a timeout error for the active execution budget."
            raise TimeoutError(f"execution exceeded budget_secs={seconds}")

        old_handler = signal.signal(signal.SIGALRM, _raise_timeout)
        old_timer = signal.setitimer(signal.ITIMER_REAL, float(seconds))
        try:
            yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
            if old_timer[0] > 0:
                signal.setitimer(signal.ITIMER_REAL, old_timer[0], old_timer[1])

    @contextmanager
    def _execution_budget(seconds, explicit):
        "Apply a time budget or report unsupported explicit timeouts."
        if not _time_limit_supported():
            if explicit:
                raise RuntimeError(
                    "execution timeout unsupported: signal timers require the main thread"
                )
            yield
            return
        with _time_limit(seconds):
            yield

    def _execute_source(source, header, filename, ns, budget_secs=None):
        "Execute source in a namespace and format captured feedback."
        budget = _bounded_int(
            budget_secs,
            session.timeout,
            1,
            _AGENT_MAX_TIMEOUT_SECONDS,
            "budget_secs",
        )
        out, err = StringIO(), StringIO()
        status, display, tb = "ok", None, ""
        try:
            with (
                _execution_budget(budget, budget_secs is not None),
                _CAPTURE_LOCK,
                redirect_stdout(out),
                redirect_stderr(err),
            ):
                display = _exec_python(source, filename, ns)
        except BaseException:
            status = "error"
            tb = traceback.format_exc()
        chunks = [f"{header} status={status} budget_secs={budget}"]
        if out.getvalue():
            chunks.append("stdout:\n" + out.getvalue().rstrip())
        if err.getvalue():
            chunks.append("stderr:\n" + err.getvalue().rstrip())
        if display is not None:
            chunks.append("display:\n" + repr(display))
        if tb:
            chunks.append("traceback:\n" + tb.rstrip())
        return "\n".join(chunks)

    def run_code(
        source: str, notebook: str | None = None, budget_secs: int | None = None
    ) -> str:
        "Run scratch Python in the live notebook scope without editing the notebook."
        path = session.target_path(notebook)
        budget = _bounded_int(
            budget_secs,
            session.timeout,
            1,
            _AGENT_MAX_TIMEOUT_SECONDS,
            "budget_secs",
        )
        source = str(source or "")
        session.record_tool(
            "run_code", f"notebook={path}, chars={len(source)}, budget_secs={budget}"
        )
        report = _execute_source(
            source,
            f"SCRATCH notebook={path}",
            f"{path}::<scratch>",
            session.live_state(path),
            budget_secs,
        )
        status = "error" if " status=error" in report else "ok"
        session.record_event(
            "scratch", tool="run_code", notebook=str(path), status=status,
            chars=len(source), budget_secs=budget,
        )
        session.record(f"Ran scratch code in {path} ({status})")
        return report

    def inspect_state(name: str | None = None, notebook: str | None = None) -> str:
        "Inspect variables and functions in the live notebook scope."
        path = session.target_path(notebook)
        ns = session.live_state(path)
        target = _none_if_blank(name)
        session.record_tool("inspect_state", f"notebook={path}, name={target!r}")
        if target is None:
            names = sorted(key for key in ns if not key.startswith("__"))
            lines = [f"state notebook={path} names={len(names)}"]
            lines.extend(f"- {key}: {type(ns[key]).__name__}" for key in names)
            session.record_event(
                "inspect", tool="inspect_state", notebook=str(path), status="ok"
            )
            return "\n".join(lines)
        if target not in ns:
            session.record_event(
                "inspect", tool="inspect_state", notebook=str(path), status="missing",
                name=target,
            )
            return f"state notebook={path} name={target} status=missing"
        value = ns[target]
        value_type = type(value)
        lines = [
            f"state notebook={path} name={target} status=ok",
            f"type={value_type.__module__}.{value_type.__qualname__}",
            "repr=" + cap_text(repr(value), 2000),
        ]
        if callable(value):
            try:
                lines.append(f"signature={target}{inspect.signature(value)}")
            except (TypeError, ValueError):
                pass
        try:
            source = inspect.getsource(value)
        except (OSError, TypeError):
            source = ""
        if source:
            lines.append("source:\n" + cap_text(source, 4000))
        session.record_event(
            "inspect", tool="inspect_state", notebook=str(path), status="ok",
            name=target,
        )
        return "\n".join(lines)

    def _execute_one(path, idx, cell, budget_secs=None):
        "Execute one notebook cell or return a skipped-cell report."
        source = cell_source(cell)
        header = f"CELL {idx} id={cell.id} notebook={path}"
        if cell.cell_type != "code":
            return f"{header} status=skipped reason=non-code"
        if _eval_false(source):
            return f"{header} status=skipped reason=eval-false"
        return _execute_source(
            source,
            header,
            f"{path}::{cell.id}",
            session.live_state(path),
            budget_secs,
        )

    def execute_cell(
        id: str,
        rerun_all: bool = False,
        notebook: str | None = None,
        budget_secs: int | None = None,
    ) -> str:
        "Execute through one cell in the live notebook state."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            target_idx, _ = _edit_find_cell_by_id(nb.cells, id)
            cells = list(nb.cells)
        if rerun_all:
            session.reset_live_state(path)
        start = session.executed_until(path) + 1
        if target_idx < start:
            start = target_idx
        budget = _bounded_int(
            budget_secs,
            session.timeout,
            1,
            _AGENT_MAX_TIMEOUT_SECONDS,
            "budget_secs",
        )
        session.record_tool(
            "execute_cell",
            (
                f"notebook={path}, id={id!r}, rerun_all={rerun_all}, "
                f"budget_secs={budget}, start={start}, target={target_idx}"
            ),
        )
        reports, status = [], "ok"
        for idx in range(start, target_idx + 1):
            report = _execute_one(path, idx, cells[idx], budget_secs=budget_secs)
            reports.append(report)
            session.set_executed_until(path, idx)
            if " status=error" in report:
                status = "error"
                break
        if not reports:
            reports.append(
                f"CELL {target_idx} id={id} notebook={path} "
                "status=skipped reason=already-executed"
            )
        session.record_event(
            "execute", tool="execute_cell", notebook=str(path), status=status,
            cell_id=id, rerun_all=rerun_all, start=start, target=target_idx,
            budget_secs=budget,
        )
        session.record(f"Executed {path} cells {start}:{target_idx} ({status})")
        return "\n\n".join([f"status={status}", *reports])

    def query_knowledge(query: str, top_k: int = 3) -> str:
        "Search the reference knowledge base."
        session.record_tool("query_knowledge", f"top_k={top_k}")
        return _knowledge_context(query, top_k=top_k)

    def query_problem_memory(query: str, top_k: int = 5, tags: str = "") -> str:
        "Search stored problem-solution memories before choosing an approach."
        top_k = _bounded_int(top_k, 5, 1, 10, "top_k")
        session.record_tool("query_problem_memory", f"top_k={top_k}")
        try:
            result = problem_statement_query(query, top_k=top_k, tags=tags)
        except BaseException as exc:
            detail = f"{type(exc).__name__}: {exc}"
            session.record_event("problem_memory_query", status="error", error=detail)
            return "Problem memory unavailable: " + detail
        session.record_event("problem_memory_query", status="ok", count=result.get("count", 0))
        return result.get("context") or "No similar problem memories found."

    def record_problem_solution(
        problem: str,
        solution: str,
        evidence: str = "",
        outcome: str = "applied",
        tags: str = "",
    ) -> str:
        "Record a reusable problem-solution pair from this task."
        session.record_tool("record_problem_solution", f"outcome={outcome}")
        task = "; ".join(_bounded_items(session.log, 8))
        try:
            result = problem_statement_add(
                problem, solution, task=task, project=str(Path.cwd()),
                evidence=evidence, outcome=outcome, tags=tags,
            )
        except BaseException as exc:
            detail = f"{type(exc).__name__}: {exc}"
            session.record_event("problem_memory_recorded", status="error", error=detail)
            return "Problem memory record failed: " + detail
        session.record_event(
            "problem_memory_recorded", status="ok", item_id=result["item_id"],
            created=result["created"], outcome=outcome,
        )
        return "recorded problem_memory item_id=" + result["item_id"]

    return [
        str_replace,
        edit_cell,
        add_cell,
        delete_cell,
        run_code,
        inspect_state,
        execute_cell,
        query_knowledge,
        query_problem_memory,
        record_problem_solution,
    ]

#### Safe persistent agent runtime

The agent now uses one audited Python runtime per notebook session. Scratch and
cell execution keep their live state, but effectful code first becomes an exact
approval request. Any notebook mutation resets that runtime so examples cannot
validate stale definitions.

In [ ]:
#| export
def _agent_runtime_capabilities(source):
    "Return effectful capabilities visible in one Python source string."
    try: tree = ast.parse(source)
    except SyntaxError: return {"python.invalid"}
    caps = set()
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call): continue
        name = ast.unparse(node.func) if hasattr(ast, "unparse") else ""
        if name == "open" and len(node.args) > 1:
            mode = getattr(node.args[1], "value", "")
            if any(flag in str(mode) for flag in "wax+"): caps.add("filesystem.write")
        if any(name.endswith("." + item) for item in (
            "write_text", "write_bytes", "touch", "mkdir", "unlink", "rmdir", "rename", "replace",
            "remove", "rmtree",
        )): caps.add("filesystem.write")
        if name.startswith(("subprocess.", "os.system", "os.popen")):
            caps.add("subprocess")
        if name.startswith(("httpx.", "requests.", "urllib.")):
            caps.add("network")
        if name.startswith(("aiosqlite.", "asyncpg.", "psycopg.", "psycopg2.", "pymongo.", "redis.", "sqlalchemy.", "sqlite3.")):
            caps.add("database")
    return caps

In [ ]:
#| export
def _agent_runtime_output_text(output):
    "Render one safe-shell output without exposing notebook JSON."
    if isinstance(output, dict):
        if "data" in output:
            data = output["data"]
            if isinstance(data, dict): return "display:\n" + str(data.get("text/plain", data))
            return "display:\n" + str(data)
        if output.get("text"): return str(output["text"])
        if output.get("ename"): return output["ename"] + ": " + str(output.get("evalue", ""))
    return str(getattr(output, "text", "") or getattr(output, "data", "") or output)

In [ ]:
#| export
class AgentRuntime:
    "One persistent audited Python runtime for an edit session notebook."
    def __init__(self, path):
        self.path, self.shell, self.journal = Path(path), None, []

    def reset(self):
        "Drop stale Python state after an earlier notebook cell changes."
        self.shell = None

    def _shell(self):
        if self.shell is None:
            from nbskill.execute import _SafeShell
            self.shell = _SafeShell(self.path)
        return self.shell

    def run(self, source, timeout, allowed=()):
        "Run source in the persistent safe shell or return an approval request."
        caps = _agent_runtime_capabilities(source)
        blocked = sorted(caps - set(allowed))
        digest = source_hash(source, length=None)
        if blocked:
            return {
                "status": "waiting_for_approval", "capabilities": blocked,
                "source_hash": digest, "source": source,
            }
        shell = self._shell()
        outputs = shell.run(source, timeout=timeout)
        status = "error" if shell.exc else "ok"
        text = "\n".join(_agent_runtime_output_text(item) for item in outputs if item).strip()
        if status == "ok": self.journal.append({"source": source, "timeout": timeout, "source_hash": digest})
        return {"status": status, "output": text, "error": str(shell.exc or "")}

In [ ]:
#| export
def _agent_runtime_timeout_risk(source):
    "Reject obvious unbounded loops before they can trap the persistent runtime."
    try: tree = ast.parse(source)
    except SyntaxError: return False
    return any(isinstance(node, ast.While) and isinstance(node.test, ast.Constant) and node.test.value is True for node in ast.walk(tree))

In [ ]:
#| export
class AgentRuntime(AgentRuntime):
    "Persistent safe runtime with an explicit guard for statically endless code."
    def run(self, source, timeout, allowed=()):
        if _agent_runtime_timeout_risk(source):
            return {"status": "error", "output": "", "error": f"TimeoutError: refused obvious unbounded loop (budget_secs={timeout})"}
        return super().run(source, timeout, allowed=allowed)

In [ ]:
#| export
def _session_runtime(session, path):
    "Return the persistent safe runtime for `path` in one agent session."
    runtimes = getattr(session, "agent_runtimes", None)
    if runtimes is None:
        runtimes = session.agent_runtimes = {}
    key = session._state_key(path)
    if key not in runtimes: runtimes[key] = AgentRuntime(path)
    return runtimes[key]

In [ ]:
#| export
def _session_reset_runtime(session, path=None):
    "Invalidate one runtime after source changes, or every runtime on reset."
    runtimes = getattr(session, "agent_runtimes", {})
    if path is None:
        for runtime in runtimes.values(): runtime.reset()
        return
    runtime = runtimes.get(session._state_key(path))
    if runtime is not None: runtime.reset()

In [ ]:
#| export
def _session_approval(session, path, source, capabilities):
    "Store one exact approval request and return its stable public shape."
    requests = getattr(session, "approval_requests", None)
    if requests is None: requests = session.approval_requests = {}
    source_hash_value = source_hash(source, length=None)
    approval_id = source_hash(f"{path}|{source_hash_value}|{','.join(capabilities)}", length=16)
    request = {
        "approval_id": approval_id, "notebook": str(path),
        "source_hash": source_hash_value, "capabilities": list(capabilities),
        "status": "waiting_for_approval",
    }
    requests[approval_id] = {**request, "source": source}
    session.pending_approval = request
    session.record_event("approval_requested", **request)
    return request

In [ ]:
#| export
def approve_agent_session(session, approval_id, decision="deny"):
    "Apply one approval decision without changing the requested source hash."
    request = getattr(session, "approval_requests", {}).get(str(approval_id))
    if request is None: raise ValueError(f"Unknown approval_id {approval_id!r}")
    decision = str(decision or "deny").lower()
    if decision not in {"once", "deny"}:
        raise ValueError("decision must be once or deny")
    if decision == "once":
        allowed = getattr(session, "approved_capabilities", set())
        allowed.add("approval:" + request["approval_id"])
        session.approved_capabilities = allowed
    request["decision"] = decision
    session.pending_approval = None
    session.record_event("approval_decided", approval_id=approval_id, decision=decision)
    return request

In [ ]:
#| export
def _session_allowed_capabilities(session, source):
    "Return capabilities approved for this source or the whole session."
    allowed = set(getattr(session, "approved_capabilities", set()))
    caps = {item for item in allowed if not item.startswith("approval:")}
    digest = source_hash(source, length=None)
    for request in getattr(session, "approval_requests", {}).values():
        if request.get("source_hash") == digest and "approval:" + request["approval_id"] in allowed:
            caps.update(request["capabilities"])
    return caps

In [ ]:
#| export
_edit_interactive_finish_write = _finish_write

In [ ]:
#| export
def _finish_write(session, path, message, result, tool_name=None):
    "Discard stale runtime before recording the successful notebook mutation."
    if isinstance(result, dict) and result.get("changed"):
        _session_reset_runtime(session, path)
    return _edit_interactive_finish_write(session, path, message, result, tool_name=tool_name)

In [ ]:
#| export
_edit_interactive_make_edit_tools = make_edit_tools

In [ ]:
#| export
def make_edit_tools(session):
    "Create notebook tools with one persistent audited runtime per target notebook."
    existing = {tool.__name__: tool for tool in _edit_interactive_make_edit_tools(session)}

    def _safe_run(path, source, header, budget_secs=None):
        budget = _bounded_int(budget_secs, session.timeout, 1, _AGENT_MAX_TIMEOUT_SECONDS, "budget_secs")
        runtime = _session_runtime(session, path)
        result = runtime.run(source, budget, allowed=_session_allowed_capabilities(session, source))
        if result["status"] == "waiting_for_approval":
            request = _session_approval(session, path, source, result["capabilities"])
            text = header + " status=waiting_for_approval\n" + json.dumps(request, sort_keys=True)
            session.session_note(text)
            return StopResponse(text)
        chunks = [f"{header} status={result['status']} budget_secs={budget}"]
        if result.get("output"): chunks.extend(["output:", result["output"]])
        if result.get("error"): chunks.extend(["error:", result["error"]])
        return "\n".join(chunks)

    def run_code(source: str, notebook: str | None=None, budget_secs: int | None=None) -> str:
        "Run scratch Python in the persistent audited notebook runtime."
        path, source = session.target_path(notebook), str(source or "")
        session.record_tool("run_code", f"notebook={path}, chars={len(source)}")
        report = _safe_run(path, source, f"SCRATCH notebook={path}", budget_secs)
        status = "waiting_for_approval" if "waiting_for_approval" in str(report) else ("error" if " status=error" in str(report) else "ok")
        if status == "ok": session.live_state(path).update(_session_runtime(session, path)._shell().g)
        session.record_event("scratch", tool="run_code", notebook=str(path), status=status, chars=len(source))
        session.record(f"Ran safe scratch code in {path} ({status})")
        return report

    def inspect_state(name: str | None=None, notebook: str | None=None) -> str:
        "Inspect variables and functions in the persistent audited runtime."
        path, target = session.target_path(notebook), _none_if_blank(name)
        ns = _session_runtime(session, path)._shell().g
        session.record_tool("inspect_state", f"notebook={path}, name={target!r}")
        if target is None:
            names = sorted(key for key in ns if not key.startswith("__"))
            result = "\n".join([f"state notebook={path} names={len(names)}", *[f"- {key}: {type(ns[key]).__name__}" for key in names]])
        elif target not in ns:
            result = f"state notebook={path} name={target} status=missing"
        else:
            value = ns[target]
            result = "\n".join([f"state notebook={path} name={target} status=ok", f"type={type(value).__module__}.{type(value).__qualname__}", "repr=" + cap_text(repr(value), 2000)])
        session.record_event("inspect", tool="inspect_state", notebook=str(path), status="ok" if "status=missing" not in result else "missing", name=target)
        return result

    def execute_cell(id: str, rerun_all: bool=False, notebook: str | None=None, budget_secs: int | None=None) -> str:
        "Execute through one cell using the persistent audited runtime."
        path = session.target_path(notebook)
        with notebook_locks(path):
            nb = read_nb(path)
            target_idx, _ = _edit_find_cell_by_id(nb.cells, id)
            cells = list(nb.cells)
        if rerun_all: _session_reset_runtime(session, path); session.set_executed_until(path, -1)
        start = session.executed_until(path) + 1
        if target_idx < start: start = target_idx
        reports, status = [], "ok"
        for idx in range(start, target_idx + 1):
            cell = cells[idx]
            source = cell_source(cell)
            if cell.cell_type != "code" or re.match(r"\s*#\|\s*eval:\s*false\b", source, re.I):
                reports.append(f"CELL {idx} id={cell.id} notebook={path} status=skipped")
                continue
            report = _safe_run(path, source, f"CELL {idx} id={cell.id} notebook={path}", budget_secs)
            reports.append(str(report))
            if "waiting_for_approval" in str(report): status = "waiting_for_approval"; break
            session.set_executed_until(path, idx)
            if " status=error" in str(report): status = "error"; break
        session.record_event("execute", tool="execute_cell", notebook=str(path), status=status, cell_id=id, rerun_all=rerun_all)
        return "\n\n".join([f"status={status}", *reports])

    def inspect_project(target: str, verbose: bool=False) -> str:
        "Read bounded notebook or symbol context outside the active chapter when needed."
        from nbskill.read import file_context
        session.record_tool("inspect_project", target)
        text = cap_text(capture_call_text(file_context, path=target, verbose=verbose), 8000)
        tag = "project:" + source_hash(target, length=10)
        session.managed_context.append(ManagedContextMessage(tag, "external project context", text, visible=True))
        session.update_context_view()
        return text

    def inspect_library(query: str, package: str="", version: str="", top_k: int=3) -> str:
        "Inspect indexed library code; downloads require the network capability."
        from nbskill.knowledge import reference_query
        source = f"library:{package}:{version}:{query}"
        wants_download = bool(package)
        allowed = _session_allowed_capabilities(session, source)
        if wants_download and "network" not in allowed:
            request = _session_approval(session, session.path, source, ["network"])
            text = "library inspection status=waiting_for_approval\n" + json.dumps(request, sort_keys=True)
            session.session_note(text)
            return StopResponse(text)
        result = reference_query(query, top_k=_bounded_int(top_k, 3, 1, 5, "top_k"), package=package or None, version=version or None, allow_download=wants_download)
        report = result.get("context") or "No relevant library context found."
        tag = "library:" + source_hash(source, length=10)
        session.managed_context.append(ManagedContextMessage(tag, "external library report", cap_text(report, 8000), visible=True))
        session.record_tool("inspect_library", f"package={package or 'indexed'}")
        session.update_context_view()
        return report

    return [
        existing["str_replace"], existing["edit_cell"], existing["add_cell"], existing["delete_cell"],
        run_code, inspect_state, execute_cell, existing["query_knowledge"],
        existing["query_problem_memory"], existing["record_problem_solution"],
        inspect_project, inspect_library,
    ]

#### Safe runtime in practice

The same tool belt can show an experiment, inspect its live value, and stop before an effectful operation. The output below is the contract an agent sees.

## Plan execution and durable sessions

The final section assembles the tools into a resumable plan and records the evidence needed to review it.

In [ ]:
visible_session = EditSession(path="nbs/08_edit_interactive.ipynb", session_id="visible-runtime-demo")
visible_tools = {tool.__name__: tool for tool in make_edit_tools(visible_session)}
print(visible_tools["run_code"]("observed = 7"))
print(visible_tools["inspect_state"]("observed"))
print(visible_tools["run_code"]("from pathlib import Path\nPath('blocked-by-policy.txt').write_text('x')"))

In [ ]:
#| hide
assert callable(make_edit_tools)

In [ ]:
#| export
def make_context_tools(session):
    "Create managed-context tools for a separate context session."

    def open_context(tag: str) -> str:
        "Open a folded managed-context message."
        return apply_context_command(session, {"action": "open", "tag": tag})

    def fold_context(tag: str, summary: str) -> str:
        "Fold a managed-context message to a concise summary."
        return apply_context_command(
            session, {"action": "fold", "tag": tag, "content": summary}
        )

    def delete_context(tag: str) -> str:
        "Delete a managed-context message from active renders."
        return apply_context_command(session, {"action": "delete", "tag": tag})

    def edit_context(tag: str, content: str) -> str:
        "Replace one managed-context message with custom content."
        return apply_context_command(
            session, {"action": "edit", "tag": tag, "content": content}
        )

    def context_view() -> str:
        "Return the currently active managed context."
        session.record_context_command({"action": "view", "tag": "managed-context"})
        return render_managed_context(session.managed_context)

    def manage_context(commands: str) -> str:
        "Apply multiple context commands in one tool call. Pass a JSON list of {action, tag, content?} dicts."
        items = json.loads(commands) if isinstance(commands, str) else commands
        if not isinstance(items, list): raise ValueError("commands must be a JSON list of dicts")
        results = []
        for cmd in items:
            if not isinstance(cmd, dict): raise ValueError(f"Each command must be a dict, got {type(cmd).__name__}")
            results.append(str(apply_context_command(session, cmd)))
        return "\n".join(results)

    return [open_context, fold_context, delete_context, edit_context, context_view, manage_context]

In [ ]:
#| export
def make_chat(model, tools, hist, system_prompt=EDIT_INTERACTIVE_SYSTEM):
    "Create the Lisette chat object for edit-interactive."
    from lisette import Chat

    return Chat(
        model, sp=system_prompt, tools=tools, hist=hist, stream=str(model).startswith("chatgpt/")
    )

In [ ]:
#| export
def run_context_session(session, model, prompt, max_steps=4, max_context_commands=8):
    "Run one separate context-management step using batched tool calls."
    if not session.managed_context: return ""
    max_steps = _bounded_int(max_steps, 4, 0, _AGENT_MAX_CONTEXT_STEPS, "max_steps")
    if max_steps == 0: return ""
    prior_chat, prior_context_idx = session.chat, session.context_msg_idx
    hist = [{"role": "user", "content": render_managed_context(session.managed_context)}]
    chat = make_chat(model, tools=make_context_tools(session), hist=hist, system_prompt=CONTEXT_SESSION_SYSTEM)
    session.chat = chat
    session.context_msg_idx = len(chat.hist) - 1
    try:
        result = chat(prompt, max_steps=max_steps, return_all=True)
    except BaseException as exc:
        summary = f"context session failed: {type(exc).__name__}: {exc}"
    else:
        summary = _bounded_text(response_text(result).strip() or "(no context response)")
    finally:
        session.chat, session.context_msg_idx = prior_chat, prior_context_idx
    session.record_message("context", summary)
    return summary

In [ ]:
#| export
def response_text(response):
    "Extract readable text from a Lisette response or response list."
    if isinstance(response, list) and response: response = response[-1]
    try:
        message = response.choices[0].message
        content = message.content
    except (AttributeError, IndexError, TypeError):
        return "" if response is None else str(response)
    if isinstance(content, list):
        return "".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return "" if content is None else str(content)

In [ ]:
#| export
def plan_result_text(result):
    "Return the human-readable text for an execute_plan result."
    if not isinstance(result, dict): return "" if result is None else str(result)
    if result.get("text"): return str(result["text"])
    sections = [
        "edit-interactive complete",
        "",
        "Final response:",
        str(result.get("summary") or "(no final response)"),
        "",
        "Tools used:",
        "\n".join(item.get("detail", item.get("tool", "")) for item in result.get("history", [])) or "(no tool calls)",
    ]
    return "\n".join(sections).rstrip()

In [ ]:
#| export
def final_diff(path):
    "Return a nbdev code-cell diff, or a clear unavailable message."
    try:
        with notebook_locks(path):
            return cap_text(capture_call_text(diff_nb, path=str(path)), _AGENT_MAX_DIFF_CHARS)
    except BaseException as exc:
        detail = str(exc)
        if "Could not find notebook" in detail or "No git repository found" in detail:
            return (
                "Code-cell diff against HEAD is unavailable because this notebook has no git baseline. "
                "This is expected for new or untracked notebooks."
            )
        return f"Code-cell diff unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def final_diffs(paths):
    "Return code-cell diffs for one or more notebooks."
    chunks = []
    for path in _target_paths(paths):
        chunks.extend([f"## {path}", final_diff(path).strip()])
    return cap_text("\n\n".join(chunks).rstrip(), _AGENT_MAX_DIFF_CHARS)

#### Running a plan

`execute_plan` packages the user's plan, the current notebook view, and the notebook tools into a Lisette chat. The result includes the final answer, tool log, operation log, revision, and code-cell diff.

In [ ]:
#| export
def _new_agent_session_id():
    "Return a short unique in-memory agent session id."
    return f"session-{uuid.uuid4().hex[:8]}"

In [ ]:
#| export
def _get_agent_session(paths, timeout, session_id=None, reset_session=False):
    "Create or reuse the persistent edit session for target notebooks."
    sid = str(session_id or "").strip() or _new_agent_session_id()
    if reset_session:
        _AGENT_SESSIONS.pop(sid, None)
    session = _AGENT_SESSIONS.get(sid)
    if session is None:
        session = EditSession(
            path=paths,
            target_paths=paths,
            timeout=timeout,
            session_id=sid,
            managed_context=managed_notebook_context(paths, revision=0),
        )
        _AGENT_SESSIONS[sid] = session
        return session
    session.target_paths = [Path(path) for path in paths]
    session.path = session.target_paths[0]
    session.timeout = timeout
    session.chat = None
    session.notebook_msg_idx = None
    session.context_msg_idx = None
    session.managed_context = managed_notebook_context(
        session.target_paths, session.revision, previous=session.managed_context
    )
    return session

In [ ]:
#| export
def _feedback_final_prompt():
    "Return the stop prompt used after mutation feedback."
    return (
        "A notebook or context update was applied. Stop this tool loop now and "
        "briefly summarize the update. The next feedback round will continue."
    )

In [ ]:
#| export
def _call_feedback_chat(chat, prompt, max_steps):
    "Call a chat with feedback-stop support and compatibility fallback."
    try:
        return chat(
            prompt,
            max_steps=max_steps,
            return_all=True,
            final_prompt=_feedback_final_prompt(),
        )
    except TypeError as exc:
        if "final_prompt" not in str(exc): raise
        return chat(prompt, max_steps=max_steps, return_all=True)

In [ ]:
#| export
def _normalize_chat_result(result):
    "Materialize streamed chat results into ordinary response values."
    if not isinstance(result, (list, str, bytes, dict)) and hasattr(result, "__next__"):
        return list(result)
    return result

In [ ]:
#| export
def execute_plan(
    notebook: str | list,  # Target notebook path, comma-separated paths, or path list
    plan: str,  # Concrete notebook-editing task for the inner agent
    model: str | None = None,  # Optional Lisette/OpenAI model name
    max_steps: int = 8,  # Maximum tool steps per feedback round
    timeout: int = 30,  # Default execution budget in seconds
    dry_run: bool = False,  # Render plan/context without invoking the agent
    symbols: str | None = None,  # Optional comma-separated symbols for impact context
    injected_context: str | None = None,  # Optional precomputed project context
    context_steps: int = 2,  # Maximum context-management steps before/after editing
    max_context_commands: int = _AGENT_MAX_CONTEXT_COMMANDS,  # Maximum streamed context commands per context step
    session_id: str | None = None,  # Reuse an in-memory edit session when provided
    reset_session: bool = False,  # Drop an existing in-memory session first
    feedback_rounds: int = 3,  # Maximum mutation feedback rounds
) -> dict:
    "Execute `plan` against one or more notebooks using bounded subagent steps."
    paths = _target_paths(notebook)
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise ValueError(f"Notebook does not exist: {', '.join(missing)}")
    max_steps = _bounded_int(max_steps, 8, 1, _AGENT_MAX_STEPS, "max_steps")
    timeout = _bounded_int(timeout, 30, 1, _AGENT_MAX_TIMEOUT_SECONDS, "timeout")
    context_steps = _bounded_int(
        context_steps, 2, 0, _AGENT_MAX_CONTEXT_STEPS, "context_steps"
    )
    feedback_rounds = _bounded_int(
        feedback_rounds, 3, 1, _AGENT_MAX_FEEDBACK_ROUNDS, "feedback_rounds"
    )
    max_context_commands = _bounded_int(
        max_context_commands, _AGENT_MAX_CONTEXT_COMMANDS, 0,
        _AGENT_MAX_CONTEXT_COMMANDS, "max_context_commands"
    )
    session_key = str(session_id or "").strip() or _new_agent_session_id()
    model = model or os.environ.get("NBSKILL_AGENT") or "chatgpt/gpt-5.4-mini"
    project_context = _bounded_text(
        injected_context or _subagent_context(paths[0], plan, symbols=symbols),
        _AGENT_MAX_PROJECT_CONTEXT_CHARS,
    )
    initial_messages = managed_notebook_context(paths, revision=0)
    initial_view = cap_text(
        render_managed_context(initial_messages), _AGENT_MAX_NOTEBOOK_VIEW_CHARS
    )
    if dry_run:
        text = "\n".join(
            [
                "notebook subagent dry run",
                "",
                f"Notebooks: {_target_label(paths)}",
                f"Model: {model}",
                f"Session id: {session_key}",
                f"Max steps: {max_steps}",
                f"Feedback rounds: {feedback_rounds}",
                f"Max context commands: {max_context_commands}",
                "Plan:",
                plan,
                "",
                "Initial managed notebook context:",
                initial_view,
                "",
                "Injected project context:",
                project_context,
            ]
        ).rstrip()
        return {
            "summary": "Dry run only; no subagent was invoked and no notebook edits were made.",
            "history": [],
            "context_commands": [],
            "feedback_notes": [],
            "feedback_rounds": 0,
            "session_id": session_key,
            "text": cap_text(text, _AGENT_MAX_NOTEBOOK_VIEW_CHARS),
        }
    session = _get_agent_session(
        paths, timeout, session_id=session_key, reset_session=reset_session
    )
    prep = (
        "Prepare context for the edit step. Open only chapters likely needed for this plan.\n"
        "Plan:\n" + plan
    )
    run_context_session(
        session, model, prep, max_steps=context_steps,
        max_context_commands=max_context_commands,
    )
    session.consume_feedback()
    initial_view = cap_text(
        render_managed_context(session.managed_context), _AGENT_MAX_NOTEBOOK_VIEW_CHARS
    )
    recent_feedback = "\n\n".join(session.feedback_notes[-3:]).strip()
    hist = [
        {"role": "user", "content": f"Plan:\n{plan}"},
        {"role": "user", "content": initial_view},
        {"role": "user", "content": "Injected project context:\n" + project_context},
    ]
    if recent_feedback:
        hist.append({"role": "user", "content": "Recent feedback notes:\n" + recent_feedback})
    for item in hist:
        session.record_message(item["role"], item["content"])
    chat = make_chat(
        model,
        tools=make_edit_tools(session) + make_context_tools(session),
        hist=hist,
    )
    session.chat = chat
    session.context_msg_idx = 1
    session.refresh_view()
    prompt = (
        "Execute the plan with the problem-solving mindset and the scratch, inspect, "
        "function, example, and test loop from the system prompt. Be ready to "
        "delete and rewrite scoped code when that makes the result clearer. First call "
        "query_problem_memory for similar old problems, then compare the old solution "
        "with your current idea before editing. Use run_code for experiments, "
        "inspect_state for live state checks, and execute_cell to validate the final "
        "example or test when practical. Use manage_context to batch context "
        "operations when you need to open, fold, or prune the active notebook view. "
        "Before finishing, call record_problem_solution for reusable lessons from "
        "this task. Stop when the notebook change is complete."
    )
    round_summaries = []
    for round_idx in range(feedback_rounds):
        session.pending_feedback = ""
        session.record_message("user", prompt)
        try:
            result = _call_feedback_chat(chat, prompt, max_steps=max_steps)
        except BaseException as exc:
            result = f"notebook subagent failed: {type(exc).__name__}: {exc}"
        result = _normalize_chat_result(result)
        summary = _bounded_text(response_text(result).strip() or "(no final response)")
        round_summaries.append(summary)
        session.record_message("assistant", summary)
        feedback = session.consume_feedback()
        if not feedback or round_idx == feedback_rounds - 1:
            break
        prompt = (
            f"Feedback round {round_idx + 2} of {feedback_rounds}.\n"
            "Last feedback note:\n"
            f"{feedback}\n\n"
            "Continue from this updated notebook/context. If the change looks "
            "wrong, fix it; otherwise test or move to the next plan step."
        )
    summary = round_summaries[-1] if round_summaries else "(no final response)"
    if session.revision and not any(
        event.get("kind") == "problem_memory_recorded" and event.get("status") == "ok"
        for event in session.events
    ):
        memory_prompt = (
            "Record reusable problem-solution memories from this task now. For each "
            "concrete problem you solved, call record_problem_solution with concise "
            "problem, solution, evidence, and tags. If nothing reusable was learned, "
            "call record_problem_solution once with outcome='skipped' and explain why."
        )
        session.record_message("user", memory_prompt)
        try:
            memory_result = _call_feedback_chat(chat, memory_prompt, max_steps=min(max_steps, 3))
        except BaseException as exc:
            memory_result = f"problem memory recording failed: {type(exc).__name__}: {exc}"
        memory_result = _normalize_chat_result(memory_result)
        memory_summary = _bounded_text(response_text(memory_result).strip() or "(no memory response)")
        session.record_message("assistant", memory_summary)
        session.consume_feedback()
    cleanup = (
        "Clean up context after the edit step. Fold useful full chapters to summaries, "
        "delete irrelevant scratch context from active renders, and leave the index visible."
    )
    run_context_session(
        session, model, cleanup, max_steps=context_steps,
        max_context_commands=max_context_commands,
    )
    session.pending_feedback = ""
    for path in paths:
        _save_notebook(read_nb(path), path)
    session.record("Exported notebook(s) after subagent run")
    diff = final_diffs(paths).strip()
    memory_events = [
        event for event in _bounded_items(session.events)
        if event.get("kind") == "problem_memory_recorded"
    ]
    memory_lines = [
        f"- {event.get('status')} {event.get('item_id', '')} outcome={event.get('outcome', '')}".rstrip()
        for event in memory_events
    ]
    text = "\n".join(
        [
            "notebook subagent complete",
            "",
            f"Session id: {session.session_id}",
            f"Feedback rounds completed: {len(round_summaries)}",
            "",
            "Final response:",
            summary,
            "",
            "Feedback notes:",
            "\n\n".join(_bounded_items(session.feedback_notes))
            if session.feedback_notes
            else "(no feedback notes)",
            "",
            "Tools used:",
            "\n".join(_bounded_items(session.tool_log))
            if session.tool_log
            else "(no tool calls)",
            "",
            "Context commands:",
            "\n".join(
                f"r{item['revision']}: {item['action']} {item['tag']} ({item['status']})"
                for item in _bounded_items(session.context_command_log)
            )
            if session.context_command_log
            else "(no context commands)",
            "",
            "Operation log:",
            "\n".join(_bounded_items(session.log))
            if session.log
            else "(no notebook operations)",
            "",
            "Problem memories:",
            "\n".join(memory_lines) if memory_lines else "(no problem memories recorded)",
            "",
            f"Final revision: {session.revision}",
            f"Agent log: {session.log_path}",
            "",
            "Notebook diff:",
            diff,
        ]
    ).rstrip()
    return {
        "summary": summary,
        "history": _bounded_items(session.history),
        "events": _bounded_items(session.events),
        "context_commands": _bounded_items(session.context_command_log),
        "feedback_notes": _bounded_items(session.feedback_notes),
        "feedback_rounds": len(round_summaries),
        "session_id": session.session_id,
        "messages": _bounded_items(session.messages),
        "text": cap_text(text, _AGENT_MAX_NOTEBOOK_VIEW_CHARS),
        "model": model,
        "notebook": str(paths[0]),
        "notebooks": [str(path) for path in paths],
        "revision": session.revision,
        "operations": _bounded_items(session.log),
        "log_path": str(session.log_path),
        "diff": diff,
    }

In [ ]:
#| export
def _agent_session_file(session_id):
    "Return the durable local checkpoint path for one agent session."
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "-", str(session_id))
    return Path("log") / f"agent-session-{safe}.json"

In [ ]:
#| export
def _checkpoint_agent_session(session):
    "Persist resumable agent reasoning, context, approvals, and safe replay journal."
    runtimes = getattr(session, "agent_runtimes", {})
    data = {
        "session_id": session.session_id,
        "target_paths": [str(path) for path in session.target_paths],
        "timeout": session.timeout, "revision": session.revision,
        "log": session.log, "tool_log": session.tool_log, "history": session.history,
        "events": session.events, "messages": session.messages,
        "feedback_notes": session.feedback_notes,
        "managed_context": [vars(msg).copy() for msg in session.managed_context],
        "context_command_log": session.context_command_log,
        "approval_requests": getattr(session, "approval_requests", {}),
        "approved_capabilities": sorted(getattr(session, "approved_capabilities", set())),
        "runtime_journal": {key: runtime.journal for key, runtime in runtimes.items()},
    }
    dest = _agent_session_file(session.session_id)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(data, indent=2, default=str), encoding="utf-8")
    return dest

In [ ]:
#| export
def _restore_agent_session(paths, timeout, session_id):
    "Restore an agent checkpoint; replay only prior pure runtime commands."
    path = _agent_session_file(session_id)
    if not path.exists(): return None
    try: data = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError): return None
    if [str(item) for item in paths] != data.get("target_paths"): return None
    session = EditSession(path=paths, target_paths=paths, timeout=timeout, session_id=session_id)
    for name in ("log", "tool_log", "history", "events", "messages", "feedback_notes", "context_command_log"):
        setattr(session, name, list(data.get(name, [])))
    session.revision = int(data.get("revision", 0))
    session.managed_context = [ManagedContextMessage(**item) for item in data.get("managed_context", [])]
    session.approval_requests = dict(data.get("approval_requests", {}))
    session.approved_capabilities = set(data.get("approved_capabilities", []))
    for key, journal in data.get("runtime_journal", {}).items():
        runtime = _session_runtime(session, Path(key))
        for item in journal:
            source = item.get("source", "")
            if not _agent_runtime_capabilities(source): runtime.run(source, item.get("timeout", timeout))
    return session

In [ ]:
#| export
_edit_interactive_get_agent_session = _get_agent_session

In [ ]:
#| export
def _get_agent_session(paths, timeout, session_id=None, reset_session=False):
    "Reuse an in-memory session or restore its durable checkpoint before planning."
    sid = str(session_id or "").strip() or _new_agent_session_id()
    if reset_session:
        _AGENT_SESSIONS.pop(sid, None)
        _agent_session_file(sid).unlink(missing_ok=True)
    session = _AGENT_SESSIONS.get(sid) or _restore_agent_session(paths, timeout, sid)
    if session is None:
        session = _edit_interactive_get_agent_session(paths, timeout, session_id=sid)
    else:
        _AGENT_SESSIONS[sid] = session
        session.target_paths, session.path, session.timeout = [Path(path) for path in paths], Path(paths[0]), timeout
        session.chat = None
        session.notebook_msg_idx = session.context_msg_idx = None
        if not session.managed_context:
            session.managed_context = managed_notebook_context(session.target_paths, session.revision)
    _checkpoint_agent_session(session)
    return session

In [ ]:
#| export
def agent_session(session_id, action="status", approval_id=None, decision="deny"):
    "Inspect, approve, deny, or restore a durable edit-interactive session."
    sid = str(session_id)
    session = _AGENT_SESSIONS.get(sid)
    if session is None:
        checkpoint = _agent_session_file(sid)
        if checkpoint.exists():
            data = json.loads(checkpoint.read_text(encoding="utf-8"))
            paths = [Path(item) for item in data.get("target_paths", [])]
            session = _restore_agent_session(paths, int(data.get("timeout", 30)), sid)
            if session is not None: _AGENT_SESSIONS[sid] = session
        if session is None: raise ValueError("Unknown agent session; resume with the original session id.")
    action = str(action or "status").lower()
    if action == "approve":
        if approval_id is None: raise ValueError("approval_id is required for action='approve'")
        approve_agent_session(session, approval_id, decision)
    elif action == "cancel":
        session.pending_approval = None
        session.record_event("session_cancelled", status="ok")
    elif action != "status":
        raise ValueError("action must be status, approve, or cancel")
    return {
        "session_id": session.session_id,
        "status": "waiting_for_approval" if getattr(session, "pending_approval", None) else "ready",
        "approval": getattr(session, "pending_approval", None),
        "checkpoint": str(_checkpoint_agent_session(session)),
        "events": _bounded_items(session.events),
    }

In [ ]:
#| hide
assert callable(agent_session)

In [ ]:
#| export
# `_split_notebooks` is defined near the notebook rendering helpers because both context and execution use it.

In [ ]:
#| export
def execute_project_plan(
    plan: str,  # Concrete notebook-editing task for the inner agent
    notebooks: str | None = None,  # Comma-separated target notebook paths
    model: str | None = None,  # Optional Lisette/OpenAI model name
    max_steps: int = 8,  # Maximum tool steps per feedback round
    timeout: int = 30,  # Default execution budget in seconds
    dry_run: bool = True,  # Render plan/context without invoking the agent by default
    symbols: str | None = None,  # Optional comma-separated symbols for impact context
    max_context_commands: int = 8,  # Maximum context commands per context session
    context_steps: int = 2,  # Maximum context-management steps before/after editing
    session_id: str | None = None,  # Reuse an in-memory edit session when provided
    reset_session: bool = False,  # Drop an existing in-memory session first
    feedback_rounds: int = 3,  # Maximum mutation feedback rounds
) -> str:
    "Launch one edit-interactive session that can touch multiple notebooks."
    targets = _split_notebooks(notebooks)
    if not targets: raise ValueError("Pass one or more notebooks to execute_project_plan.")
    result = execute_plan(
        notebook=targets, plan=plan, model=model, max_steps=max_steps,
        timeout=timeout, dry_run=dry_run, symbols=symbols,
        max_context_commands=max_context_commands, context_steps=context_steps,
        session_id=session_id, reset_session=reset_session,
        feedback_rounds=feedback_rounds,
    )
    return "\n".join([
        "project subagent coordinator", "", f"Dry run: {dry_run}", f"Targets: {len(targets)}", "", plan_result_text(result),
    ]).rstrip()

## Examples and tests

The examples below keep outputs visible; contract tests are hidden so they validate the notebook without crowding the rendered page.

In [ ]:
#| hide
with write_demo_notebook("08_edit_view.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp sample"),
        mk_cell("#| export\ndef public():\n    return 1"),
        mk_cell("assert public() == 1"),
    ]), path)
    view = ei.notebook_view(path, revision=3)
    assert "Revision: 3" in view
    assert "type=code" in view
    assert "type=code" in view
    assert "public()" in view

In [ ]:
#| hide
with write_demo_notebook("08_edit_tools.ipynb") as path:
    first = mk_cell("#| default_exp sample")
    second = mk_cell("x = 1")
    _write_tmp_nb(new_nb([first, second]), path)
    session = ei.EditSession(path=path)

    class FakeChat:
        "Small fake chat used by edit-tool tests."
        def __init__(self):
            "Seed fake chat history for the test."
            self.hist = [
                {"role": "user", "content": "plan"},
                {"role": "user", "content": ei.notebook_view(path)},
            ]

    session.chat = FakeChat()
    session.notebook_msg_idx = 1
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    replaced = tools["str_replace"]("x = 1", "x = 10")
    assert "Replaced text" in replaced
    added = tools["add_cell"](second.id, "y = 2")
    nb = _read_tmp_nb(path)
    assert "Added cell" in added
    assert len(nb.cells) == 3
    assert "y = 2" in session.chat.hist[1]["content"]
    new_id = nb.cells[-1].id
    edited = tools["edit_cell"](new_id, "2", "3")
    assert "Edited text" in edited
    nb = _read_tmp_nb(path)
    assert nb.cells[-1].source == "y = 3"
    removed = tools["delete_cell"](new_id)
    assert "Deleted cell" in removed
    assert len(_read_tmp_nb(path).cells) == 2
    assert session.revision == 4
    assert session.log_path.exists()

    assert "status=ok" in tools["run_code"]("scratch_x = 1")
    state = tools["inspect_state"]("scratch_x")
    assert "status=ok" in state and "repr=1" in state
    assert "display:\n2" in tools["run_code"]("scratch_x + 1")
    timeout_report = tools["run_code"]("while True:\n    pass", budget_secs=1)
    assert (
        "TimeoutError" in timeout_report
        or "execution exceeded budget_secs=1" in timeout_report
        or "execution timeout unsupported" in timeout_report
    )

with write_demo_notebook("08_edit_execution_reset.ipynb") as path:
    setup = mk_cell("x = 1")
    expr = mk_cell("x + 1")
    _write_tmp_nb(new_nb([setup, expr]), path)
    session = ei.EditSession(path=path)
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    assert "display:\n2" in tools["execute_cell"](expr.id)
    tools["edit_cell"](setup.id, "x = 1", "x = 10")
    assert "display:\n11" in tools["execute_cell"](expr.id)

In [ ]:
#| hide
with write_demo_notebook("08_edit_scratch_event.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = ei.EditSession(path=path)
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    scratch = tools["run_code"]("probe = 3\nprobe")

assert "status=ok" in scratch
assert session.events[-1]["kind"] == "scratch"
assert session.events[-1]["status"] == "ok"

In [ ]:
#| hide
with write_demo_notebook("08_edit_inspect_event.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = ei.EditSession(path=path)
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    missing = tools["inspect_state"]("missing_name")

assert "status=missing" in missing
assert session.events[-1]["kind"] == "inspect"
assert session.events[-1]["status"] == "missing"

In [ ]:
#| hide
with write_demo_notebook("08_edit_write_event.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = ei.EditSession(path=path)
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    mutation = tools["add_cell"](None, "2 + 1")

assert "Empirical next step" in mutation
assert session.events[-1]["kind"] == "write"
assert session.events[-1]["tool"] == "add_cell"

In [ ]:
#| hide
with write_demo_notebook("08_edit_execute_event.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = ei.EditSession(path=path)
    tools = {tool.__name__: tool for tool in ei.make_edit_tools(session)}
    tools["add_cell"](None, "2 + 1")
    new_id = read_nb(path).cells[-1].id
    report = tools["execute_cell"](new_id)

assert "status=ok" in report
assert session.events[-1]["kind"] == "execute"
assert session.events[-1]["status"] == "ok"

In [ ]:
#| hide
with write_demo_notebook("08_edit_managed_context.ipynb") as path:
    intro = mk_cell(
        "## Intro\nThis chapter explains the public answer.", cell_type="markdown"
    )
    body = mk_cell("#| export\ndef public_answer():\n    return 42")
    details = mk_cell("## Details\nExtra notes live here.", cell_type="markdown")
    _write_tmp_nb(new_nb([intro, body, details]), path)
    session = ei.EditSession(path=path)
    session.managed_context = ei.managed_notebook_context(path)

    class FakeContextChat:
        "Small fake chat used by managed-context tests."
        def __init__(self):
            "Seed fake chat history for the context view."
            self.hist = [{"role": "user", "content": ""}]

    session.chat = FakeContextChat()
    session.context_msg_idx = 0
    rendered = ei.render_managed_context(session.managed_context)
    tag = f"chapter:{intro.id}"
    assert tag in rendered
    assert "public_answer" in rendered
    assert "return 42" not in rendered
    assert "Folded messages:" in rendered

    opened = ei.apply_context_command(session, {"action": "open", "tag": tag})
    assert isinstance(opened, StopResponse)
    assert "Notebook/context updated" in str(opened)
    assert session.pending_feedback == str(opened)
    assert session.messages[-1]["role"] == "note"
    assert session.chat.hist[-1]["content"] == str(opened)
    assert "return 42" in session.chat.hist[0]["content"]
    folded = ei.apply_context_command(
        session, {"action": "fold", "tag": tag, "content": "intro folded"}
    )
    assert isinstance(folded, StopResponse)
    folded_view = session.chat.hist[0]["content"]
    assert "intro folded" in folded_view
    assert "return 42" not in folded_view
    ei.apply_context_command(session, {"action": "open", "tag": tag})
    add_cell = ei.make_edit_tools(session)[2]
    add_cell(body.id, "added_value = 3")
    assert "added_value = 3" in session.chat.hist[0]["content"]
    assert "return 42" in session.chat.hist[0]["content"]
    ei.apply_context_command(
        session, {"action": "edit", "tag": tag, "content": "custom context"}
    )
    assert "custom context" in session.chat.hist[0]["content"]
    ei.apply_context_command(session, {"action": "delete", "tag": tag})
    assert "custom context" not in session.chat.hist[0]["content"]
    assert session.context_command_log[-1]["action"] == "delete"

with write_demo_notebook("08_edit_context_stream.ipynb") as path:
    intro = mk_cell("## Stream\nContext command target.", cell_type="markdown")
    _write_tmp_nb(new_nb([intro]), path)
    session = ei.EditSession(path=path)
    session.managed_context = ei.managed_notebook_context(path)
    tag = f"chapter:{intro.id}"

    class ClosingStream:
        "Closable iterable stream used by context-command tests."
        def __init__(self, chunks):
            "Store chunks and track whether close was called."
            self.chunks, self.closed = chunks, False
        def __iter__(self):
            "Iterate through streamed chunks."
            return iter(self.chunks)
        def close(self):
            "Mark the stream as closed."
            self.closed = True

    class StreamingChat:
        "Fake streaming chat that emits one context command."
        model = "fake"
        def __init__(self):
            "Seed stream and prompt capture for the test."
            self.hist = [{"role": "user", "content": ""}]
            self.streams = []
            self.prompts = []
        def __call__(self, prompt, max_steps=8, stream=False, **kwargs):
            "Return a command stream first and a completion stream after feedback."
            self.prompts.append(prompt)
            if "Do not repeat" in prompt:
                stream_obj = ClosingStream(["done"])
            else:
                stream_obj = ClosingStream([
                    "before ",
                    f"[[ctx:fold {tag}]]stream summary[[/ctx:fold]]",
                    " after",
                ])
            self.streams.append(stream_obj)
            return stream_obj

    chat = StreamingChat()
    session.chat = chat
    session.context_msg_idx = 0
    text = ei.run_chat_with_context_commands(session, "prepare", max_context_commands=1)
    assert text == "before done"
    assert chat.streams[0].closed
    assert isinstance(session.feedback_notes[-1], str)
    assert "context fold applied" in session.feedback_notes[-1]
    assert "stream summary" in session.chat.hist[0]["content"]
    assert "Notebook/context updated" in chat.prompts[-1]

fold_cmd = ei.find_context_command("[[ctx:fold demo]]short summary[[/ctx:fold]]")
assert fold_cmd["action"] == "fold"
assert fold_cmd["tag"] == "demo"
assert fold_cmd["content"] == "short summary"
assert ei.find_context_command("[[ctx:delete demo]]")["action"] == "delete"
edit_cmd = ei.find_context_command("[[ctx:edit demo]]replacement[[/ctx:edit]]")
assert edit_cmd["action"] == "edit"
assert edit_cmd["content"] == "replacement"

with write_demo_notebook("08_edit_manage_context.ipynb") as path:
    intro = mk_cell("## Intro\nBatched context target.", cell_type="markdown")
    details = mk_cell("## Details\nMore info here.", cell_type="markdown")
    _write_tmp_nb(new_nb([intro, details]), path)
    session = ei.EditSession(path=path)
    session.managed_context = ei.managed_notebook_context(path)
    session.chat = FakeContextChat()
    session.context_msg_idx = 0
    tag_intro, tag_details = f"chapter:{intro.id}", f"chapter:{details.id}"
    tools = ei.make_context_tools(session)
    manage = [t for t in tools if t.__name__ == "manage_context"][0]
    result = manage(json.dumps([{"action": "open", "tag": tag_intro}, {"action": "fold", "tag": tag_details, "content": "details summary"}]))
    assert "context open applied" in result
    assert "context fold applied" in result
    view = session.chat.hist[0]["content"]
    assert "Batched context target" in view
    assert "details summary" in view
    assert len([c for c in session.context_command_log if c["action"] in ("open", "fold")]) >= 2

In [ ]:
#| hide
with write_demo_notebook("08_edit_update.ipynb") as path:
    cell = mk_cell("x = 1")
    _write_tmp_nb(new_nb([cell]), path)
    session = ei.EditSession(path=path)
    edit_cell = ei.make_edit_tools(session)[1]
    result = edit_cell(cell.id, "1", "2")
    assert f"Edited text in {path} id={cell.id}" in result
    assert read_nb(path).cells[0].source == "x = 2"
    stale = edit_cell(cell.id, "x = 1", "x = 3")
    assert '"status": "no_change"' in stale
    assert '"matches": 0' in stale
    assert "source_hash" in stale
    assert "current_source:\nx = 2" in stale
    assert read_nb(path).cells[0].source == "x = 2"

In [ ]:
#| hide
with write_demo_notebook("08_edit_missing.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    delete_cell = ei.make_edit_tools(session)[3]
    try:
        delete_cell("missing")
    except ValueError as exc:
        assert "No cell has id" in str(exc)
    else:
        raise AssertionError("expected missing cell failure")

In [ ]:
#| hide
with write_demo_notebook("08_edit_duplicate.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1"), mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    str_replace = ei.make_edit_tools(session)[0]
    try:
        str_replace("x = 1", "x = 2")
    except ValueError as exc:
        assert "matched 2 cells" in str(exc)
    else:
        raise AssertionError("expected ambiguous replace failure")

In [ ]:
#| hide
from contextlib import contextmanager

In [ ]:
#| hide
@contextmanager
def _patched_make_chat(fake):
    "Temporarily replace make_chat in edit-interactive tests."
    old_make_chat = ei.make_chat
    ei.make_chat = fake
    try:
        yield
    finally:
        ei.make_chat = old_make_chat

In [ ]:
#| hide
class FakeChat:
    "Fake chat that exercises execute_plan feedback rounds."
    calls = []

    def __init__(self, model, sp, tools, hist):
        "Capture initialization arguments and available tools."
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist
        FakeChat.calls.append(
            {"kind": "init", "tools": [tool.__name__ for tool in tools], "system": sp}
        )

    def __call__(
        self, msg, max_steps=20, return_all=False, stream=False, final_prompt=None
    ):
        "Drive context setup, mutation, and feedback validation in tests."
        names = [tool.__name__ for tool in self.tools]
        by_name = {tool.__name__: tool for tool in self.tools}
        FakeChat.calls.append(
            {"kind": "call", "tools": names, "msg": msg, "stream": stream}
        )
        if "Prepare context" in msg and "open_context" in names:
            by_name["open_context"](stream_tag)
            return "context step done"
        if "Clean up context" in msg and "fold_context" in names:
            by_name["fold_context"](stream_tag, "demo folded")
            return "context step done"
        if "Record reusable problem-solution" in msg and "record_problem_solution" in names:
            by_name["record_problem_solution"](
                "Need to validate notebook edit feedback",
                "Execute the changed cell after feedback",
                evidence="fake execute_plan test",
                tags="feedback,notebook,testing,solution-category",
            )
            return "memory recorded"
        assert "add_cell" in names
        if "Feedback round" in msg:
            new_id = read_nb(path).cells[-1].id
            report = by_name["execute_cell"](new_id)
            assert "status=ok" in report
            return "validated after feedback"
        if "context_view" in names:
            by_name["context_view"]()
        mutation = by_name["add_cell"](None, "answer = 42")
        assert isinstance(mutation, StopResponse)
        assert "Notebook/context updated" in mutation
        return "mutation applied"

In [ ]:
#| hide
def fake_make_chat(model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM):
    "Return the feedback-round fake chat for execute_plan tests."
    return FakeChat(model, system_prompt, tools, hist)

In [ ]:
#| hide
with write_demo_notebook("08_edit_plan.ipynb") as path:
    absolute_path = path.resolve()
    first = mk_cell("## Demo\nA tiny managed chapter.", cell_type="markdown")
    _write_tmp_nb(new_nb([first]), absolute_path)
    stream_tag = f"chapter:{first.id}"
    agent_args = dict(
        model="fake", max_steps=2, timeout=2, injected_context="test context",
        feedback_rounds=2,
    )
    FakeChat.calls = []
    with _patched_make_chat(fake_make_chat):
        result = ei.execute_plan(
            str(absolute_path), "Inspect the demo chapter.", **agent_args
        )
    text = ei.plan_result_text(result)

assert result["summary"] == "validated after feedback"
assert result["feedback_rounds"] == 2
assert result["feedback_notes"]
assert result["context_commands"][0]["action"] == "open"
assert result["context_commands"][0]["tag"] == stream_tag
assert result["context_commands"][-1]["action"] == "fold"
assert any(
    call["kind"] == "init"
    and "open_context" in call["tools"]
    and "fold_context" in call["tools"]
    for call in FakeChat.calls
)
assert any(
    call["kind"] == "init"
    and "add_cell" in call["tools"]
    and "run_code" in call["tools"]
    and "query_problem_memory" in call["tools"]
    and "record_problem_solution" in call["tools"]
    and "open_context" in call["tools"]
    for call in FakeChat.calls
)
assert any("Feedback round 2" in call.get("msg", "") for call in FakeChat.calls)
assert "Final response:" in text
assert "Tools used:" in text
assert "Agent log:" in text
assert "validated after feedback" in text
assert "Problem memories:" in text
assert any(item["tool"] == "add_cell" for item in result["history"])
assert any(item["kind"] == "write" for item in result["events"])
assert result["events"][0]["tool"] == "add_cell"
assert result["events"]
assert result["events"]
assert result["revision"] == 1
assert result["log_path"].endswith(".log")

with write_demo_notebook("08_edit_multi_a.ipynb") as path_a:
    with write_demo_notebook("08_edit_multi_b.ipynb") as path_b:
        a = mk_cell("# A\nFirst", cell_type="markdown")
        b = mk_cell("# B\nSecond", cell_type="markdown")
        _write_tmp_nb(new_nb([a, mk_cell("x = 1")]), path_a)
        _write_tmp_nb(new_nb([b, mk_cell("y = 2")]), path_b)
        messages = ei.managed_notebook_context([path_a, path_b])
        tags = [msg.tag for msg in messages if msg.tag != "notebook:index"]
        assert all(tag.startswith("notebook:") for tag in tags)
        session = ei.EditSession(path=[path_a, path_b], managed_context=messages)
        str_replace = ei.make_edit_tools(session)[0]
        try:
            str_replace("x = 1", "x = 3")
        except ValueError as exc:
            assert "notebook is required" in str(exc)
        else:
            raise AssertionError("expected explicit notebook target")
        str_replace("x = 1", "x = 3", notebook=path_a.name)
        assert "x = 3" in ei.notebook_view(path_a)
        assert "y = 2" in ei.notebook_view(path_b)


class PersistentChat:
    "Fake chat that verifies persistent session reuse."
    def __init__(self, model, sp, tools, hist):
        "Capture initialization arguments for persistent-session tests."
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist

    def __call__(
        self, msg, max_steps=20, return_all=False, stream=False, final_prompt=None
    ):
        "Store or inspect live state depending on the persisted plan."
        names = [tool.__name__ for tool in self.tools]
        by_name = {tool.__name__: tool for tool in self.tools}
        plan_text = self.hist[0]["content"] if self.hist else ""
        if "Prepare context" in msg and "fold_context" in names:
            by_name["fold_context"](persist_tag, "persist folded")
            return "context folded"
        if "Clean up context" in msg:
            return "context clean"
        if "Persist first" in plan_text:
            report = by_name["run_code"]("persist_value = 41")
            assert "status=ok" in report
            return "persist stored"
        if "Persist second" in plan_text:
            state = by_name["inspect_state"]("persist_value")
            assert "repr=41" in state
            view = by_name["context_view"]()
            assert "persist folded" in view
            return "persist inspected"
        return "noop"

In [ ]:
#| hide
def fake_persistent_make_chat(
    model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM
):
    "Return the persistent-session fake chat for execute_plan tests."
    return PersistentChat(model, system_prompt, tools, hist)

In [ ]:
#| hide
with write_demo_notebook("08_edit_persistent.ipynb") as path:
    absolute_path = path.resolve()
    chapter = mk_cell("## Persist\nA reusable context chapter.", cell_type="markdown")
    _write_tmp_nb(new_nb([chapter]), absolute_path)
    persist_tag = f"chapter:{chapter.id}"
    session_key = "persist-feedback-loop"
    ei._AGENT_SESSIONS.pop(session_key, None)
    with _patched_make_chat(fake_persistent_make_chat):
        first_result = ei.execute_plan(
            str(absolute_path), "Persist first", model="fake", max_steps=1,
            timeout=2, injected_context="test", session_id=session_key,
            reset_session=True, feedback_rounds=1,
        )
        second_result = ei.execute_plan(
            str(absolute_path), "Persist second", model="fake", max_steps=1,
            timeout=2, injected_context="test", session_id=session_key,
            feedback_rounds=1,
        )
    persisted = ei._AGENT_SESSIONS[session_key]
    assert persisted.live_state(absolute_path)["persist_value"] == 41
    assert any(
        msg.tag == persist_tag and msg.summary == "persist folded"
        for msg in persisted.managed_context
    )

assert first_result["session_id"] == session_key
assert second_result["session_id"] == session_key
assert second_result["summary"] == "persist inspected"

In [ ]:
#| hide
with write_demo_notebook("08_edit_safe_runtime_scratch.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = EditSession(path=path, session_id="safe-runtime-scratch-test")
    session.managed_context = managed_notebook_context(path)
    _AGENT_SESSIONS[session.session_id] = session
    tools = {tool.__name__: tool for tool in make_edit_tools(session)}
    assert "status=ok" in tools["run_code"]("remembered = 42")
    assert "repr=42" in tools["inspect_state"]("remembered")
    _AGENT_SESSIONS.pop(session.session_id, None)

#### Durable session control

`agent_session` is the small control surface for a paused run: inspect its checkpoint, approve the pending effect, or cancel it before the agent continues.

In [ ]:
#| eval: false
agent_session("session-id", action="status")

In [ ]:
#| hide
with write_demo_notebook("08_edit_safe_runtime_approval.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = EditSession(path=path, session_id="safe-runtime-approval-test")
    session.managed_context = managed_notebook_context(path)
    _AGENT_SESSIONS[session.session_id] = session
    tools = {tool.__name__: tool for tool in make_edit_tools(session)}
    blocked = tools["run_code"]("from pathlib import Path\nPath('blocked.txt').write_text('x')")
    database = tools["run_code"]("import sqlite3\nsqlite3.connect('live.db')")
    assert "status=ok" in tools["run_code"]("remembered = 42")
    assert "waiting_for_approval" in str(blocked)
    assert '"database"' in str(database)
    request = session.pending_approval
    assert agent_session(session.session_id, "approve", request["approval_id"], "once")["status"] == "ready"
    checkpoint = _agent_session_file(session.session_id)
    checkpoint.unlink()
    _AGENT_SESSIONS.pop(session.session_id, None)

In [ ]:
#| hide
with write_demo_notebook("08_edit_safe_runtime_restore.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("value = 1")]), path)
    session = EditSession(path=path, session_id="safe-runtime-restore-test")
    session.managed_context = managed_notebook_context(path)
    _AGENT_SESSIONS[session.session_id] = session
    tools = {tool.__name__: tool for tool in make_edit_tools(session)}
    assert "status=ok" in tools["run_code"]("remembered = 42")
    checkpoint = _checkpoint_agent_session(session)
    _AGENT_SESSIONS.pop(session.session_id, None)
    restored = agent_session(session.session_id)
    restored_tools = {tool.__name__: tool for tool in make_edit_tools(_AGENT_SESSIONS[session.session_id])}
    assert restored["status"] == "ready"
    assert "repr=42" in restored_tools["inspect_state"]("remembered")
    checkpoint.unlink()
    _AGENT_SESSIONS.pop(session.session_id, None)